# Historical case study: completing documented constructions

Companion to the paper **Knowledge-Constrained Neural Completion of Islamic Geometric Patterns with Symmetry Guarantees**.

The quantitative experiments in the paper use procedurally generated patterns. This notebook provides the complementary qualitative case study on control geometry transcribed from documented construction practice: the ring proportions of three classical rosette constructions, at symmetry orders eight, ten, and twelve, are entered by hand from the polygonal-technique literature, and the trained orbit-tied model completes each one. No quantitative claim is made. The purpose is to show that the knowledge-constrained completion mechanism operates on externally specified, tradition-derived control geometry exactly as it does on the procedural training distribution, producing completions that are exactly symmetric by construction and verified by the label-free rotation audit.

The transcribed proportions below are entered as editable parameters, so any documented construction can be studied by changing them. Run top to bottom; CPU is sufficient and no training occurs.

## Demo setup

In [ ]:
# ============================================================
# Demo setup: locate the repository and load shared paths
# ============================================================
# This notebook lives in demo/ inside the repository. When run locally it finds
# the repository root automatically. When run on Google Colab it clones the
# repository first so the shipped checkpoints and cached tables are available.
from pathlib import Path
import os, sys, platform, json, math, random, shutil, csv, time, copy
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Any, Optional
import numpy as np

def in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False

IN_COLAB = in_colab()
print('Python:', sys.version.split()[0], '| Colab:', IN_COLAB)

def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / 'Results' / 'models').exists():
            return cand
    return here

CODE_ROOT = find_repo_root()
if IN_COLAB and not (CODE_ROOT / 'Results' / 'models').exists():
    os.system('git clone https://github.com/ugail/Symmetry-Structured-IGP-Completion.git /content/igp_repo')
    CODE_ROOT = Path('/content/igp_repo')

DATASET_ROOT   = CODE_ROOT / 'wikimedia_igp_dataset'
RESULTS_ROOT   = CODE_ROOT / 'Results'
FIG_DIR        = RESULTS_ROOT / 'figures'
METRIC_DIR     = RESULTS_ROOT / 'metrics'
SVG_DIR        = RESULTS_ROOT / 'svg'
MODEL_DIR      = RESULTS_ROOT / 'models'
EXTERNAL_OUT_DIR = RESULTS_ROOT / 'external_sanity'
DEMO_OUT       = CODE_ROOT / 'demo' / 'outputs'
for d in [FIG_DIR, METRIC_DIR, SVG_DIR, MODEL_DIR, EXTERNAL_OUT_DIR, DEMO_OUT]:
    d.mkdir(parents=True, exist_ok=True)
print('Repository:', CODE_ROOT)
print('Checkpoints:', sorted(p.name for p in MODEL_DIR.glob('*.pt')) or 'NONE FOUND')

## Imports and device

In [ ]:
# ============================================================
# 1. Imports, reproducibility, and publication figure style
# ============================================================
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from PIL import Image, ImageOps, ImageDraw
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from IPython.display import display
except Exception:
    def display(x): print(x)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Torch:', torch.__version__, 'Device:', DEVICE)

def set_seed(seed: int = 0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True); return path

def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2)

def load_json(path: Path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

# ---- Consistent, professional figure style (used by every figure) ----
PALETTE = {
    'constrained':   '#1b6f8c',   # teal-blue
    'unconstrained': '#d1622b',   # warm orange
    'unc_free':      '#b23a48',   # red
    'template':      '#6f6f6f',   # grey
    'nn':            '#5c4b8a',   # purple
    'accent':        '#1d1d1d',
    'paper_bg':      '#fbfaf6',
}
def setup_plot_style():
    plt.rcParams.update({
        'figure.dpi': 120, 'savefig.dpi': 320, 'savefig.bbox': 'tight',
        'savefig.facecolor': 'white', 'figure.facecolor': 'white',
        'font.family': 'DejaVu Sans', 'font.size': 11,
        'axes.titlesize': 12.5, 'axes.titleweight': 'semibold',
        'axes.labelsize': 11.5, 'legend.fontsize': 9.5, 'legend.frameon': False,
        'xtick.labelsize': 10, 'ytick.labelsize': 10,
        'axes.spines.top': False, 'axes.spines.right': False,
        'axes.linewidth': 0.9, 'axes.grid': True, 'grid.alpha': 0.22,
        'grid.linewidth': 0.6, 'axes.axisbelow': True,
        'mathtext.default': 'regular', 'figure.constrained_layout.use': True,
    })
setup_plot_style()

def savefig_png(fig, path: Path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    if path.suffix.lower() != '.png':
        path = path.with_suffix('.png')
    fig.savefig(path, dpi=320, bbox_inches='tight', facecolor='white')
    print('Saved figure:', path)
    return path


## Configuration

In [ ]:
# ============================================================
# 2. Experiment configuration
# ============================================================
@dataclass
class Config:
    # ---- Data ----
    smoke_mode: bool = False     # True = fast code check only (NOT paper results)
    seed: int = 7
    train_n: int = 600
    val_n: int = 120
    test_n: int = 120
    smoke_train_n: int = 28
    smoke_val_n: int = 8
    smoke_test_n: int = 8

    # ---- Training (stabilized) ----
    epochs: int = 70
    smoke_epochs: int = 4
    lr: float = 7e-4              # lowered from 2e-3 to remove the mid-training divergence
    lr_min_ratio: float = 0.04   # cosine floor as a fraction of base lr
    warmup_epochs: int = 5       # linear LR warmup stabilizes the first steps
    grad_clip: float = 1.0
    weight_decay: float = 1e-5
    restore_best: bool = True    # restore the lowest-val-loss checkpoint at the end
    hidden_dim: int = 96
    smoke_hidden_dim: int = 32
    layers: int = 3

    # ---- Candidate lattice / domain ----
    symmetry_orders: Tuple[int, ...] = (6, 8, 10, 12)
    train_symmetry_orders: Tuple[int, ...] = (6, 8, 12)
    heldout_symmetry_order: int = 10
    num_regimes: int = 4
    num_families: int = 5
    heldout_family: int = 4
    heldout_regime: int = 3
    alpha_max: float = 0.16

    # ---- Losses ----
    bce_weight: float = 1.0
    density_weight: float = 0.35
    count_weight: float = 0.20
    alpha_straight_weight: float = 0.02
    alpha_l2_weight: float = 0.002
    pos_weight_max: float = 3.0

    # ---- Selection / calibration ----
    default_threshold: float = 0.5
    calibrate_selection: bool = True
    selection_mode: str = 'topk_ratio'
    threshold_grid_size: int = 41
    topk_ratio_min: float = 0.14
    topk_ratio_max: float = 0.55
    topk_ratio_grid_size: int = 42
    density_penalty: float = 1.15
    free_topk_ratio_min: float = 0.08
    free_topk_ratio_max: float = 0.42
    free_topk_ratio_grid_size: int = 36
    calibration_metric: str = 'f1_density'

    # ---- Corrupted-input training control ----
    train_augmented_variants: bool = True
    augment_max_dropout: float = 0.30

    # ---- Statistical analysis of the fidelity comparison ----
    equivalence_margin_f1: float = 0.02

    # ---- Incremental checkpoint reuse ----
    reuse_checkpoints: bool = True
    reuse_cached_ablations: bool = True

    # ---- Multi-seed protocol ----
    run_multi_seed: bool = True
    seeds: Tuple[int, ...] = (7, 11, 19)

    # ---- Per-N (out-of-distribution symmetry) evaluation ----
    per_n_test_n: int = 90        # test graphs generated per symmetry order

    # ---- Data-efficiency ablation ----
    run_data_efficiency: bool = True
    data_eff_sizes: Tuple[int, ...] = (100, 200, 400, 600)
    data_eff_epochs: int = 55     # slightly shorter to bound ablation runtime
    data_eff_seeds: Tuple[int, ...] = (7, 11)   # subset of seeds to bound runtime

    # ---- Imperfect-control-geometry robustness experiment ----
    # Corrupt the control geometry the network *reads* (asymmetric coordinate jitter) and
    # ask whether orbit-tied training still beats simply projecting an ordinary
    # model's output onto the orbit structure at inference. This is the decisive
    # test of whether the architectural constraint is necessary.
    run_robustness: bool = True
    robustness_modes: Tuple[str, ...] = ('jitter', 'dropout', 'sector')
    robustness_jitter: Tuple[float, ...] = (0.0, 0.02, 0.05, 0.10, 0.15)
    robustness_dropout: Tuple[float, ...] = (0.0, 0.10, 0.20, 0.30)
    robustness_sector: Tuple[float, ...] = (0.0, 0.02, 0.05, 0.10, 0.15)
    robustness_eval_n: int = 60          # test graphs used per (mode, severity)
    robustness_sym_points: int = 240     # points sampled for the symmetry metric

    # ---- Qualitative showcase ----
    gallery_best_of_n: int = 8
    gallery_linewidth: float = 1.45
    gallery_axis_limit: float = 1.38
    gallery_target_density_min: float = 0.20
    gallery_target_density_max: float = 0.62
    save_gallery_svgs: bool = True

CFG = Config()
if CFG.smoke_mode:
    CFG.train_n, CFG.val_n, CFG.test_n = CFG.smoke_train_n, CFG.smoke_val_n, CFG.smoke_test_n
    CFG.epochs = CFG.smoke_epochs
    CFG.warmup_epochs = 1
    CFG.hidden_dim = CFG.smoke_hidden_dim
    CFG.layers = 1
    CFG.seeds = (7,)
    CFG.per_n_test_n = 8
    CFG.data_eff_sizes = (24, 48)
    CFG.data_eff_epochs = 3
    CFG.data_eff_seeds = (7,)
    CFG.robustness_jitter = (0.0, 0.1)
    CFG.robustness_dropout = (0.0, 0.2)
    CFG.robustness_sector = (0.0, 0.1)
    CFG.robustness_eval_n = 4
    CFG.robustness_sym_points = 120

set_seed(CFG.seed)
print('PAPER MODE' if not CFG.smoke_mode else 'SMOKE MODE (code check only)')
print(CFG)


## Geometric primitives

In [ ]:
# ============================================================
# 3. Geometry helpers
# ============================================================

TWO_PI = 2 * math.pi

def rotmat(theta: float) -> np.ndarray:
    c, s = math.cos(theta), math.sin(theta)
    return np.array([[c, -s], [s, c]], dtype=np.float32)

def polar(r: float, theta: float) -> np.ndarray:
    return np.array([r * math.cos(theta), r * math.sin(theta)], dtype=np.float32)

def angle_of(p: np.ndarray) -> float:
    return math.atan2(float(p[1]), float(p[0]))

def radius_of(p: np.ndarray) -> float:
    return float(np.linalg.norm(p))

def angle_diff(a: float, b: float) -> float:
    d = (a - b + math.pi) % (2 * math.pi) - math.pi
    return d

def normalize_points(points: np.ndarray) -> np.ndarray:
    pts = points.copy()
    mn = pts.min(axis=0)
    mx = pts.max(axis=0)
    scale = max(float((mx - mn).max()), 1e-8)
    pts = (pts - (mn + mx) / 2.0) / scale
    return pts.astype(np.float32)

def unique_edges(edges: List[Tuple[int, int]]) -> List[Tuple[int, int]]:
    s = set()
    out = []
    for a, b in edges:
        if a == b:
            continue
        e = (a, b) if a < b else (b, a)
        if e not in s:
            s.add(e)
            out.append(e)
    return out

def line_segments_from_edges(points: np.ndarray, edges: List[Tuple[int, int]]) -> np.ndarray:
    if len(edges) == 0:
        return np.zeros((0, 2, 2), dtype=np.float32)
    return np.array([[points[i], points[j]] for i, j in edges], dtype=np.float32)

## Data structures

In [ ]:
# ============================================================
# 4. Scaffold, candidate lattice and target graph data classes
# ============================================================

EDGE_TYPES = {
    'ring': 0,
    'radial': 1,
    'skip': 2,
    'diag': 3,
    'boundary': 4,
    'star_cross': 5,
    'secondary': 6,
    'border': 7,
}
EDGE_TYPE_NAMES = {v: k for k, v in EDGE_TYPES.items()}
NUM_EDGE_TYPES = len(EDGE_TYPES)

@dataclass
class PatternSample:
    sample_id: str
    scaffold_type: str
    N: int
    family_id: int
    regime_id: int
    points: np.ndarray                  # [V, 2]
    node_features: np.ndarray           # [V, Fv]
    edges: np.ndarray                   # [E, 2]
    edge_features: np.ndarray           # [E, Fe]
    edge_types: np.ndarray              # [E]
    edge_labels: np.ndarray             # [E], 0/1 target
    orbit_ids: np.ndarray               # [E]
    anchor_mask: np.ndarray             # [V]
    target_edges: List[Tuple[int, int]]
    scaffold_info: Dict[str, Any]

## Control geometry and candidate lattice

In [ ]:
# ============================================================
# 5. Unified scaffold-implied candidate lattice
# ============================================================

def make_scaffold(
    N: int,
    scaffold_type: str,
    rng: np.random.Generator,
    outer_radius: float = 1.0,
) -> Dict[str, Any]:
    """Create minimal or anchored scaffold metadata.

    Minimal scaffold:
        center + symmetry order + boundary radius.
    Anchored scaffold:
        center + outer star tips + inner valley anchors + boundary.
    """
    assert scaffold_type in ['minimal', 'anchored']
    inner_ratio = float(rng.uniform(0.36, 0.58))
    mid_ratio = float(rng.uniform(0.62, 0.78))
    small_ratio = float(rng.uniform(0.20, 0.32))
    phase = float(rng.choice([0.0, math.pi / N]))

    info = {
        'N': int(N),
        'scaffold_type': scaffold_type,
        'outer_radius': float(outer_radius),
        'inner_ratio': inner_ratio,
        'mid_ratio': mid_ratio,
        'small_ratio': small_ratio,
        'phase': phase,
        'center': [0.0, 0.0],
    }
    return info

def build_candidate_lattice(scaffold: Dict[str, Any]) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, List[Dict[str, Any]]]:
    """Build a unified candidate lattice closed under C_N.

    The lattice is deliberately a superset across target families. Families are represented by
    different target masks over this same candidate space, not by separate candidate generators.
    """
    N = int(scaffold['N'])
    scaffold_type = scaffold['scaffold_type']
    outer = float(scaffold['outer_radius'])
    inner = float(scaffold['inner_ratio']) * outer
    mid = float(scaffold['mid_ratio']) * outer
    small = float(scaffold['small_ratio']) * outer

    # ring specifications: (name, radius, phase_kind)
    # phase_kind 0: sector angles i/N, phase_kind 1: half-sector angles (i+0.5)/N
    ring_specs = [
        ('small_0', small, 0),
        ('small_h', small, 1),
        ('inner_0', inner, 0),
        ('inner_h', inner, 1),
        ('mid_0', mid, 0),
        ('mid_h', mid, 1),
        ('outer_0', outer, 0),
        ('outer_h', outer, 1),
    ]

    points = []
    labels = []
    anchor_mask = []

    # center
    points.append(np.array([0.0, 0.0], dtype=np.float32))
    labels.append({'kind': 'center', 'ring': -1, 'phase': 0, 'sector': -1, 'r': 0.0, 'theta': 0.0})
    anchor_mask.append(True)

    idx_by_key = {('center', -1, 0, -1): 0}

    for ring_idx, (ring_name, r, phase_kind) in enumerate(ring_specs):
        for i in range(N):
            theta = TWO_PI * (i + 0.5 * phase_kind) / N
            idx = len(points)
            points.append(polar(r, theta))
            labels.append({'kind': ring_name, 'ring': ring_idx, 'phase': phase_kind, 'sector': i, 'r': r, 'theta': theta})
            # Anchored scaffolds have outer star tips and inner valley anchors fixed.
            is_anchor = False
            if scaffold_type == 'anchored':
                if ring_name in ['outer_0', 'inner_h']:
                    is_anchor = True
            else:
                # Minimal scaffolds only fix the boundary ring and center.
                if ring_name == 'outer_0':
                    is_anchor = True
            anchor_mask.append(is_anchor)
            idx_by_key[(ring_name, ring_idx, phase_kind, i)] = idx

    points = np.array(points, dtype=np.float32)
    anchor_mask = np.array(anchor_mask, dtype=bool)

    # Helpers
    def ring_indices_by_name(name):
        return [i for i, lab in enumerate(labels) if lab['kind'] == name]

    edges = []
    edge_types = []

    def add_edge(a, b, typ):
        if a == b:
            return
        edges.append((a, b) if a < b else (b, a))
        edge_types.append(EDGE_TYPES[typ])

    # Ring edges
    for ring_name, _, _ in ring_specs:
        inds = ring_indices_by_name(ring_name)
        inds_by_sector = {labels[idx]['sector']: idx for idx in inds}
        for i in range(N):
            add_edge(inds_by_sector[i], inds_by_sector[(i + 1) % N], 'ring')

    # Boundary edges more strongly typed
    for ring_name in ['outer_0', 'outer_h']:
        inds = ring_indices_by_name(ring_name)
        bysec = {labels[idx]['sector']: idx for idx in inds}
        for i in range(N):
            add_edge(bysec[i], bysec[(i + 1) % N], 'boundary')

    # Radial edges center-to-rings and ring-to-ring same sector/phase
    center_idx = 0
    radial_names = ['small_0', 'inner_0', 'mid_0', 'outer_0']
    radial_h_names = ['small_h', 'inner_h', 'mid_h', 'outer_h']
    for names in [radial_names, radial_h_names]:
        prev = None
        for name in names:
            bysec = {labels[idx]['sector']: idx for idx in ring_indices_by_name(name)}
            for i in range(N):
                if prev is None:
                    add_edge(center_idx, bysec[i], 'radial')
                else:
                    add_edge(prev[i], bysec[i], 'radial')
            prev = bysec

    # Skip-k star links on selected rings. k values 2 and 3 where meaningful.
    for ring_name in ['inner_0', 'inner_h', 'mid_0', 'outer_0']:
        bysec = {labels[idx]['sector']: idx for idx in ring_indices_by_name(ring_name)}
        for k in [2, 3]:
            if k < N // 2 + 1:
                for i in range(N):
                    add_edge(bysec[i], bysec[(i + k) % N], 'skip')

    # Cross-ring diagonals
    pairs = [
        ('small_0', 'inner_h'), ('small_h', 'inner_0'),
        ('inner_0', 'mid_h'), ('inner_h', 'mid_0'),
        ('mid_0', 'outer_h'), ('mid_h', 'outer_0'),
        ('inner_h', 'outer_0'), ('inner_0', 'outer_h')
    ]
    for a_name, b_name in pairs:
        A = {labels[idx]['sector']: idx for idx in ring_indices_by_name(a_name)}
        B = {labels[idx]['sector']: idx for idx in ring_indices_by_name(b_name)}
        for i in range(N):
            add_edge(A[i], B[i], 'diag')
            add_edge(A[i], B[(i + 1) % N], 'diag')

    # Star-cross connectors
    A = {labels[idx]['sector']: idx for idx in ring_indices_by_name('inner_h')}
    B = {labels[idx]['sector']: idx for idx in ring_indices_by_name('outer_0')}
    C = {labels[idx]['sector']: idx for idx in ring_indices_by_name('mid_h')}
    for i in range(N):
        add_edge(A[i], B[i], 'star_cross')
        add_edge(A[i], B[(i + 1) % N], 'star_cross')
        add_edge(C[i], B[i], 'star_cross')
        add_edge(C[i], B[(i + 1) % N], 'star_cross')

    # Secondary connectors
    for a_name, b_name in [('small_0', 'mid_0'), ('small_h', 'mid_h'), ('small_0', 'outer_0')]:
        A = {labels[idx]['sector']: idx for idx in ring_indices_by_name(a_name)}
        B = {labels[idx]['sector']: idx for idx in ring_indices_by_name(b_name)}
        for i in range(N):
            add_edge(A[i], B[(i + 1) % N], 'secondary')
            add_edge(A[i], B[(i - 1) % N], 'secondary')

    # Border connectors
    A = {labels[idx]['sector']: idx for idx in ring_indices_by_name('mid_0')}
    B = {labels[idx]['sector']: idx for idx in ring_indices_by_name('outer_h')}
    for i in range(N):
        add_edge(A[i], B[i], 'border')
        add_edge(A[i], B[(i + 1) % N], 'border')

    # Unique edges while preserving first edge type
    type_by_edge = {}
    for e, t in zip(edges, edge_types):
        if e not in type_by_edge:
            type_by_edge[e] = t
    edges_unique = list(type_by_edge.keys())
    edge_types_unique = np.array([type_by_edge[e] for e in edges_unique], dtype=np.int64)
    edges_np = np.array(edges_unique, dtype=np.int64)

    # Node features
    node_features = []
    for i, p in enumerate(points):
        r = radius_of(p)
        th = angle_of(p)
        lab = labels[i]
        node_features.append([
            p[0], p[1],
            r,
            math.cos(th), math.sin(th),
            float(anchor_mask[i]),
            (lab['ring'] + 1) / max(1, len(ring_specs)),
            float(lab['phase']),
            float(scaffold_type == 'anchored'),
            N / 12.0,
        ])
    node_features = np.array(node_features, dtype=np.float32)

    # Edge features
    edge_features = []
    for (a, b), typ in zip(edges_unique, edge_types_unique):
        pa, pb = points[a], points[b]
        ra, rb = radius_of(pa), radius_of(pb)
        ta, tb = angle_of(pa), angle_of(pb)
        d = pb - pa
        length = float(np.linalg.norm(d))
        ad = abs(angle_diff(tb, ta)) / math.pi
        rd = abs(rb - ra)
        type_oh = [0.0] * NUM_EDGE_TYPES
        type_oh[int(typ)] = 1.0
        edge_features.append([
            d[0], d[1], length, rd, ad,
            ra, rb,
            float(N) / 12.0,
            float(scaffold_type == 'anchored'),
        ] + type_oh)
    edge_features = np.array(edge_features, dtype=np.float32)

    return points, node_features, edges_np, edge_features, edge_types_unique, anchor_mask, labels

def compute_edge_orbits_from_labels(
    edges: np.ndarray,
    edge_types: np.ndarray,
    labels: List[Dict[str, Any]],
    N: int
) -> np.ndarray:
    """Compute cyclic edge orbits using lattice labels.

    Canonicalises an edge by rotating sectors so the minimum sector is 0.
    Center endpoints are handled separately.
    """
    orbit_keys = []
    for (a, b), typ in zip(edges, edge_types):
        la, lb = labels[int(a)], labels[int(b)]
        def token(lab):
            return (lab['kind'], int(lab['ring']), int(lab['phase']), int(lab['sector']))
        ta, tb = token(la), token(lb)

        sectors = [s for s in [ta[3], tb[3]] if s >= 0]
        if len(sectors) == 0:
            shift = 0
        else:
            shift = min(sectors)

        def shift_token(t):
            kind, ring, phase, sec = t
            sec2 = -1 if sec < 0 else (sec - shift) % N
            return (kind, ring, phase, sec2)

        ca, cb = shift_token(ta), shift_token(tb)
        pair = tuple(sorted([ca, cb]))
        orbit_keys.append((int(typ), pair))

    key_to_id = {}
    ids = []
    for k in orbit_keys:
        if k not in key_to_id:
            key_to_id[k] = len(key_to_id)
        ids.append(key_to_id[k])
    return np.array(ids, dtype=np.int64)

## Target patterns and samples

In [ ]:
# ============================================================
# 6. Stochastic target distribution
# ============================================================

def target_rule_prob(
    edge_type: int,
    family_id: int,
    regime_id: int,
    rng: np.random.Generator,
) -> float:
    """Template-centred stochastic target probability for an edge type.

    The target distribution must not be a purely independent Bernoulli draw over the
    whole candidate lattice. Each family/regime pair defines a coherent template over
    edge types. Stochasticity then perturbs that template at the orbit level. This
    gives controlled variation without turning the target into an arbitrary random
    subset of candidate edges.

    Family/regime names are neutral procedural regimes, not historical/cultural labels.
    """
    name = EDGE_TYPE_NAMES[int(edge_type)]

    # Coherent family templates. Values are probabilities for orbit inclusion.
    # They are deliberately sparse enough that the model must learn edge selection
    # instead of being rewarded for selecting the entire candidate lattice.
    family_templates = {
        0: {'ring': .72, 'radial': .30, 'skip': .70, 'diag': .08, 'boundary': 1.00, 'star_cross': .06, 'secondary': .12, 'border': .18},
        1: {'ring': .34, 'radial': .16, 'skip': .28, 'diag': .36, 'boundary': 1.00, 'star_cross': .78, 'secondary': .14, 'border': .20},
        2: {'ring': .55, 'radial': .22, 'skip': .38, 'diag': .10, 'boundary': 1.00, 'star_cross': .15, 'secondary': .20, 'border': .72},
        3: {'ring': .30, 'radial': .12, 'skip': .62, 'diag': .58, 'boundary': 1.00, 'star_cross': .34, 'secondary': .44, 'border': .22},
        4: {'ring': .76, 'radial': .34, 'skip': .32, 'diag': .18, 'boundary': 1.00, 'star_cross': .22, 'secondary': .58, 'border': .44},
    }

    # Regime modifications. These create controlled multi-regime synthesis from the
    # same scaffold without making unsupported historical-style claims.
    regime_mods = {
        # A: ring-dominant / calm rosette
        0: {'ring': +.18, 'radial': +.05, 'skip': -.12, 'diag': -.16, 'star_cross': -.18, 'secondary': -.10, 'border': -.06},
        # B: sharp star / radial connector emphasis
        1: {'ring': -.02, 'radial': +.18, 'skip': +.18, 'diag': -.02, 'star_cross': -.04, 'secondary': -.04, 'border': -.08},
        # C: star-cross / diagonal emphasis
        2: {'ring': -.16, 'radial': -.06, 'skip': -.08, 'diag': +.24, 'star_cross': +.32, 'secondary': -.02, 'border': -.10},
        # D: nested/border/secondary emphasis
        3: {'ring': +.04, 'radial': -.02, 'skip': +.00, 'diag': +.06, 'star_cross': +.02, 'secondary': +.26, 'border': +.26},
    }

    p = family_templates[int(family_id)].get(name, 0.05)
    p += regime_mods[int(regime_id)].get(name, 0.0)

    # Light orbit-level randomness. Keep this small: coherence comes from templates,
    # not from unstructured random edge selection.
    p += float(rng.normal(0.0, 0.020))
    return float(np.clip(p, 0.01, 1.00))


def _orbits_for_type(edge_types: np.ndarray, orbit_ids: np.ndarray, type_name: str) -> np.ndarray:
    typ = EDGE_TYPES[type_name]
    idx = np.where(edge_types == typ)[0]
    if len(idx) == 0:
        return np.array([], dtype=int)
    return np.unique(orbit_ids[idx]).astype(int)


def _activate_orbits(y: np.ndarray, orbit_ids: np.ndarray, chosen_orbits: List[int]):
    chosen = set(int(o) for o in chosen_orbits)
    if not chosen:
        return
    mask = np.array([int(o) in chosen for o in orbit_ids], dtype=bool)
    y[mask] = 1.0


def generate_target_labels(
    edge_types: np.ndarray,
    orbit_ids: np.ndarray,
    family_id: int,
    regime_id: int,
    rng: np.random.Generator,
) -> np.ndarray:
    """Generate orbit-consistent, template-centred stochastic target edge labels.

    Important design choice:
    - Every stochastic decision is made at the orbit level.
    - The fallback for sparse graphs activates whole orbits, not individual edges.
    This preserves the cyclic target symmetry needed for fair training of the
    constrained model.
    """
    y = np.zeros(len(edge_types), dtype=np.float32)
    unique_orbits = np.unique(orbit_ids).astype(int)

    # Global optional gates create coherent sample-level variation. If a gate is off,
    # the corresponding edge family is strongly suppressed throughout the graph.
    gates = {
        'diag': rng.random() < (0.24 + 0.14 * (int(family_id) in [1, 3]) + 0.36 * (int(regime_id) == 2) - 0.10 * (int(regime_id) == 0)),
        'star_cross': rng.random() < (0.18 + 0.24 * (int(family_id) == 1) + 0.42 * (int(regime_id) == 2) - 0.08 * (int(regime_id) == 0)),
        'secondary': rng.random() < (0.22 + 0.16 * (int(family_id) in [3, 4]) + 0.38 * (int(regime_id) == 3)),
        'border': rng.random() < (0.34 + 0.22 * (int(family_id) == 2) + 0.42 * (int(regime_id) == 3)),
    }

    for oid in unique_orbits:
        idxs = np.where(orbit_ids == oid)[0]
        typ = int(edge_types[idxs[0]])
        name = EDGE_TYPE_NAMES[typ]
        p = target_rule_prob(typ, family_id, regime_id, rng)

        # Coherent optional structures: suppress optional types when their sample-level
        # gate is off, rather than independently flickering many orbits on/off.
        if name in gates and not gates[name]:
            p *= 0.18

        include = rng.random() < p
        y[idxs] = 1.0 if include else 0.0

    # Boundary orbits are always active.
    boundary_orbits = _orbits_for_type(edge_types, orbit_ids, 'boundary')
    _activate_orbits(y, orbit_ids, list(boundary_orbits))

    # Ensure coherent internal structure, using whole orbits only.
    min_edges = max(8, int(round(0.10 * len(y))))
    if int(y.sum()) < min_edges:
        fallback_types = ['ring', 'skip', 'radial']
        for typ_name in fallback_types:
            oids = _orbits_for_type(edge_types, orbit_ids, typ_name)
            if len(oids) == 0:
                continue
            k = max(1, int(math.ceil(0.18 * len(oids))))
            chosen = rng.choice(oids, size=min(k, len(oids)), replace=False)
            _activate_orbits(y, orbit_ids, list(map(int, chosen)))
            if int(y.sum()) >= min_edges:
                break

    # Defensive invariant: all labels are constant within each orbit.
    # This is cheap and catches accidental target asymmetry immediately.
    for oid in unique_orbits:
        idxs = np.where(orbit_ids == oid)[0]
        if len(idxs) > 1:
            # If numerical or programming errors ever create mixed labels, use majority.
            val = 1.0 if y[idxs].mean() >= 0.5 else 0.0
            y[idxs] = val

    return y.astype(np.float32)

def make_sample(
    sample_id: str,
    N: int,
    scaffold_type: str,
    family_id: int,
    regime_id: int,
    rng: np.random.Generator,
) -> PatternSample:
    scaffold = make_scaffold(N, scaffold_type, rng)
    points, node_features, edges, edge_features, edge_types, anchor_mask, labels = build_candidate_lattice(scaffold)
    orbit_ids = compute_edge_orbits_from_labels(edges, edge_types, labels, N)
    y = generate_target_labels(edge_types, orbit_ids, family_id, regime_id, rng)
    target_edges = [tuple(map(int, e)) for e, yy in zip(edges, y) if yy > 0.5]

    info = dict(scaffold)
    info.update({'labels': labels})
    return PatternSample(
        sample_id=sample_id,
        scaffold_type=scaffold_type,
        N=N,
        family_id=family_id,
        regime_id=regime_id,
        points=points,
        node_features=node_features,
        edges=edges,
        edge_features=edge_features,
        edge_types=edge_types,
        edge_labels=y,
        orbit_ids=orbit_ids,
        anchor_mask=anchor_mask,
        target_edges=target_edges,
        scaffold_info=info,
    )

def make_dataset(
    n: int,
    seed: int,
    split: str,
    scaffold_types=('anchored', 'minimal'),
    symmetry_orders=(6, 8, 10, 12),
    families=(0,1,2,3,4),
    regimes=(0,1,2,3),
) -> List[PatternSample]:
    rng = np.random.default_rng(seed)
    data = []
    for i in range(n):
        N = int(rng.choice(symmetry_orders))
        scaffold_type = str(rng.choice(scaffold_types, p=[0.72, 0.28] if len(scaffold_types)==2 else None))
        family_id = int(rng.choice(families))
        regime_id = int(rng.choice(regimes))
        data.append(make_sample(f'{split}_{i:05d}', N, scaffold_type, family_id, regime_id, rng))
    return data

## Exact geometric orbit partition (label-free)

In [ ]:
# ============================================================
# 7b. Exact geometric orbit partition (label-free)
# ============================================================
# Candidate edges are grouped into rotational orbits by an exact geometric
# procedure rather than by symbolic labels: the C_N rotation map is computed on
# the vertex coordinates, every edge is connected to its rotation image, and the
# connected components of that relation (union-find) are the true orbits. A
# label-free audit is also provided: it rotates every selected edge by exactly
# 2*pi/N and checks that the image edge is selected, so the verification of the
# symmetry guarantee is independent of the identifiers the system itself uses.

def rotation_vertex_map(pts, N, tol=1e-4):
    c, s = math.cos(2 * math.pi / N), math.sin(2 * math.pi / N)
    R = np.array([[c, -s], [s, c]])
    rot = np.asarray(pts, dtype=np.float64) @ R.T
    m = np.full(len(pts), -1, dtype=int)
    for i, p in enumerate(rot):
        d = np.linalg.norm(np.asarray(pts, dtype=np.float64) - p, axis=1)
        j = int(np.argmin(d))
        if d[j] < tol:
            m[i] = j
    return m

def fix_orbit_ids(sample):
    """Replace label-based orbit identifiers with the exact geometric partition."""
    vm = rotation_vertex_map(sample.points, int(sample.N))
    assert (vm >= 0).all(), 'candidate lattice is not closed under C_N rotation'
    eidx = {frozenset((int(a), int(b))): k for k, (a, b) in enumerate(sample.edges)}
    parent = list(range(len(sample.edges)))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for k, (a, b) in enumerate(sample.edges):
        img = eidx[frozenset((int(vm[int(a)]), int(vm[int(b)])))]
        ra, rb = find(k), find(img)
        if ra != rb:
            parent[rb] = ra
    roots = [find(k) for k in range(len(sample.edges))]
    remap = {r: i for i, r in enumerate(dict.fromkeys(roots))}
    s2 = copy.copy(sample)
    s2.orbit_ids = np.array([remap[r] for r in roots], dtype=sample.orbit_ids.dtype)
    return s2

def true_rotation_violations(sample, selected):
    """Label-free audit: selected edges whose exact rotation image is unselected."""
    vm = rotation_vertex_map(sample.points, int(sample.N))
    eidx = {frozenset((int(a), int(b))): k for k, (a, b) in enumerate(sample.edges)}
    bad = 0
    for k, (a, b) in enumerate(sample.edges):
        if selected[k]:
            img = eidx[frozenset((int(vm[int(a)]), int(vm[int(b)])))]
            if not selected[img]:
                bad += 1
    return bad

_orig_make_sample = make_sample
def make_sample(*args, **kwargs):
    return fix_orbit_ids(_orig_make_sample(*args, **kwargs))
print('Orbit identifiers: exact geometric partition (union-find under the C_N rotation map).')


## Orbit operations

In [ ]:
# ============================================================
# 7. PyTorch conversion and orbitwise projection
# ============================================================

def sample_to_torch(sample: PatternSample, device=DEVICE) -> Dict[str, torch.Tensor]:
    return {
        'node_features': torch.tensor(sample.node_features, dtype=torch.float32, device=device),
        'edge_index': torch.tensor(sample.edges.T, dtype=torch.long, device=device),  # [2, E]
        'edge_features': torch.tensor(sample.edge_features, dtype=torch.float32, device=device),
        'edge_labels': torch.tensor(sample.edge_labels, dtype=torch.float32, device=device),
        'edge_types': torch.tensor(sample.edge_types, dtype=torch.long, device=device),
        'orbit_ids': torch.tensor(sample.orbit_ids, dtype=torch.long, device=device),
        'anchor_mask': torch.tensor(sample.anchor_mask.astype(np.float32), dtype=torch.float32, device=device),
    }

def orbit_average(values: torch.Tensor, orbit_ids: torch.Tensor) -> torch.Tensor:
    """Average values over integer orbit IDs."""
    out = torch.zeros_like(values)
    unique = torch.unique(orbit_ids)
    for oid in unique:
        mask = orbit_ids == oid
        out[mask] = values[mask].mean()
    return out

## Edge message-passing model

In [ ]:
# ============================================================
# 8. Neural model
# ============================================================

class EdgeMessagePassing(nn.Module):
    def __init__(self, node_dim: int, edge_dim: int, hidden_dim: int, layers: int = 3, num_regimes: int = 4):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.layers = layers
        self.node_in = nn.Sequential(
            nn.Linear(node_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.regime_emb = nn.Embedding(num_regimes, hidden_dim)

        self.msg_layers = nn.ModuleList()
        self.upd_layers = nn.ModuleList()
        for _ in range(layers):
            self.msg_layers.append(nn.Sequential(
                nn.Linear(hidden_dim * 2 + edge_dim, hidden_dim),
                nn.GELU(),
                nn.Linear(hidden_dim, hidden_dim),
            ))
            self.upd_layers.append(nn.Sequential(
                nn.Linear(hidden_dim * 2, hidden_dim),
                nn.GELU(),
                nn.Linear(hidden_dim, hidden_dim),
            ))

        self.edge_head = nn.Sequential(
            nn.Linear(hidden_dim * 3 + edge_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, 1),
        )
        self.alpha_head = nn.Sequential(
            nn.Linear(hidden_dim * 3 + edge_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.Tanh(),
        )

    def forward(self, batch: Dict[str, torch.Tensor], regime_id: int, constrained: bool = True):
        x = batch['node_features']
        edge_index = batch['edge_index']
        edge_attr = batch['edge_features']
        orbit_ids = batch['orbit_ids']

        src, dst = edge_index[0], edge_index[1]
        h = self.node_in(x)

        for msg, upd in zip(self.msg_layers, self.upd_layers):
            # undirected message passing: send both directions
            m1 = msg(torch.cat([h[src], h[dst], edge_attr], dim=-1))
            m2 = msg(torch.cat([h[dst], h[src], edge_attr], dim=-1))
            agg = torch.zeros_like(h)
            agg.index_add_(0, dst, m1)
            agg.index_add_(0, src, m2)
            deg = torch.zeros((h.shape[0], 1), device=h.device)
            one = torch.ones((src.shape[0], 1), device=h.device)
            deg.index_add_(0, dst, one)
            deg.index_add_(0, src, one)
            agg = agg / deg.clamp(min=1.0)
            h = h + upd(torch.cat([h, agg], dim=-1))

        z = self.regime_emb(torch.tensor([regime_id], dtype=torch.long, device=x.device)).expand(edge_attr.shape[0], -1)
        edge_context = torch.cat([h[src], h[dst], z, edge_attr], dim=-1)

        logits = self.edge_head(edge_context).squeeze(-1)
        alpha_raw = self.alpha_head(edge_context).squeeze(-1)

        if constrained:
            logits = orbit_average(logits, orbit_ids)
            alpha_raw = orbit_average(alpha_raw, orbit_ids)

        return logits, alpha_raw

## Loss

In [ ]:
# ============================================================
# 9. Loss functions
# ============================================================

def model_loss(
    logits: torch.Tensor,
    alpha_raw: torch.Tensor,
    batch: Dict[str, torch.Tensor],
    cfg: Config,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    y = batch['edge_labels']
    edge_types = batch['edge_types']

    # Class imbalance: compute per-sample pos_weight.
    pos = y.sum().clamp(min=1.0)
    neg = (1.0 - y).sum().clamp(min=1.0)
    # Too-large positive weights can push early training into the degenerate
    # "predict everything" solution. Clamp conservatively and add an explicit
    # count-matching penalty below.
    pos_weight = (neg / pos).detach().clamp(0.5, cfg.pos_weight_max)
    bce = F.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_weight)

    p = torch.sigmoid(logits)
    density = torch.abs(p.sum() - y.sum()) / y.numel()
    count_match = ((p.sum() - y.sum()) / y.numel()).pow(2)

    structural_mask = (
        (edge_types == EDGE_TYPES['ring']) |
        (edge_types == EDGE_TYPES['radial']) |
        (edge_types == EDGE_TYPES['boundary'])
    ).float()
    alpha_straight = (structural_mask * alpha_raw.pow(2)).mean()
    alpha_l2 = alpha_raw.pow(2).mean()

    loss = (
        cfg.bce_weight * bce
        + cfg.density_weight * density
        + cfg.count_weight * count_match
        + cfg.alpha_straight_weight * alpha_straight
        + cfg.alpha_l2_weight * alpha_l2
    )

    return loss, {
        'bce': float(bce.detach().cpu()),
        'density': float(density.detach().cpu()),
        'count_match': float(count_match.detach().cpu()),
        'pos_weight': float(pos_weight.detach().cpu()),
        'alpha_straight': float(alpha_straight.detach().cpu()),
        'alpha_l2': float(alpha_l2.detach().cpu()),
    }

## Metrics

In [ ]:
# ============================================================
# 10. Refinement, prediction and metrics
# ============================================================

def select_edges_from_scores(
    scores: np.ndarray,
    orbit_ids: np.ndarray,
    threshold: float = 0.5,
    topk_orbits: Optional[int] = None,
    topk_ratio: Optional[float] = None,
) -> np.ndarray:
    """Select candidate edges from scores.

    Selection is orbitwise whenever topk_orbits or topk_ratio is used. This is
    the preferred policy for the constrained synthesis experiments because it
    controls edge density and avoids the low-threshold, select-everything failure
    mode observed in early smoke runs.
    """
    orbit_ids = np.asarray(orbit_ids)
    scores = np.asarray(scores, dtype=np.float32)

    if topk_ratio is not None:
        unique = np.unique(orbit_ids)
        topk_orbits = max(1, int(round(float(topk_ratio) * len(unique))))

    if topk_orbits is not None:
        unique = np.unique(orbit_ids)
        orbit_scores = []
        for oid in unique:
            orbit_scores.append((oid, float(scores[orbit_ids == oid].mean())))
        orbit_scores.sort(key=lambda x: x[1], reverse=True)
        k = max(1, min(int(topk_orbits), len(orbit_scores)))
        chosen = set([oid for oid, _ in orbit_scores[:k]])
        return np.array([oid in chosen for oid in orbit_ids], dtype=bool)

    return scores >= threshold

def policy_to_label(policy: Dict[str, float]) -> str:
    mode = policy.get('mode')
    if mode == 'topk_ratio':
        return f"orbit topK={policy.get('topk_ratio', 0):.2f}"
    if mode == 'free_topk_ratio':
        return f"free topK={policy.get('topk_ratio', 0):.2f}"
    if mode == 'free_threshold':
        return f"free τ={policy.get('threshold', 0.5):.2f}"
    return f"τ={policy.get('threshold', 0.5):.2f}"

def select_with_policy(scores: np.ndarray, orbit_ids: np.ndarray, policy: Dict[str, float]) -> np.ndarray:
    """Select candidate edges under a named policy.

    Modes:
      threshold       : independent thresholding on edge scores.
      topk_ratio      : orbitwise top-K selection, preserving cyclic edge orbits.
      free_threshold  : unconstrained independent thresholding; used to expose
                        symmetry violations in the unconstrained baseline.
      free_topk_ratio : unconstrained individual-edge top-K selection; controls
                        density but does not force orbit closure.
    """
    mode = policy.get('mode', 'threshold')
    scores = np.asarray(scores, dtype=np.float32)
    if mode == 'topk_ratio':
        return select_edges_from_scores(scores, orbit_ids, topk_ratio=policy.get('topk_ratio', 0.35))
    if mode == 'free_threshold':
        return scores >= float(policy.get('threshold', 0.5))
    if mode == 'free_topk_ratio':
        ratio = float(policy.get('topk_ratio', 0.25))
        k = max(1, min(len(scores), int(round(ratio * len(scores)))))
        idx = np.argsort(-scores)[:k]
        out = np.zeros(len(scores), dtype=bool)
        out[idx] = True
        return out
    return select_edges_from_scores(scores, orbit_ids, threshold=policy.get('threshold', 0.5))

def calibration_objective(metrics: Dict[str, float], cfg: Config) -> float:
    """Validation objective for threshold/top-K calibration.

    Raw F1 can reward very low thresholds that select almost all edges. The
    density-aware objective penalises edge-density error, improving precision and
    figure quality without using target edges at inference time.
    """
    if cfg.calibration_metric == 'f1':
        return float(metrics['f1'])
    if cfg.calibration_metric == 'balanced':
        return float(0.5 * metrics['f1'] + 0.5 * metrics['precision_at_k'] - cfg.density_penalty * metrics['edge_density_error'])
    # default: density-aware F1
    return float(metrics['f1'] - cfg.density_penalty * metrics['edge_density_error'])

def refined_edge_polylines(
    points: np.ndarray,
    edges: np.ndarray,
    selected: np.ndarray,
    alpha: np.ndarray,
    edge_types: np.ndarray,
    alpha_max: float,
    samples_per_edge: int = 8,
) -> List[np.ndarray]:
    """Return polylines for selected edges. Structural edges remain straight."""
    polylines = []
    for idx, sel in enumerate(selected):
        if not sel:
            continue
        a, b = map(int, edges[idx])
        p0, p1 = points[a], points[b]
        typ = int(edge_types[idx])
        if typ in [EDGE_TYPES['ring'], EDGE_TYPES['radial'], EDGE_TYPES['boundary']]:
            polylines.append(np.stack([p0, p1], axis=0))
            continue
        v = p1 - p0
        L = np.linalg.norm(v) + 1e-9
        n = np.array([-v[1], v[0]], dtype=np.float32) / L
        m = 0.5 * (p0 + p1)
        aa = float(np.tanh(alpha[idx]) * alpha_max)
        q = m + aa * L * n
        ts = np.linspace(0, 1, samples_per_edge)
        curve = []
        for t in ts:
            # Quadratic Bezier through q.
            pt = (1-t)**2 * p0 + 2*(1-t)*t * q + t**2 * p1
            curve.append(pt)
        polylines.append(np.array(curve, dtype=np.float32))
    return polylines

def sample_points_from_polylines(polylines: List[np.ndarray], max_points: int = 500) -> np.ndarray:
    pts = []
    for poly in polylines:
        if len(poly) == 0:
            continue
        pts.extend(poly.tolist())
    if len(pts) == 0:
        return np.zeros((1, 2), dtype=np.float32)
    pts = np.array(pts, dtype=np.float32)
    if len(pts) > max_points:
        idx = np.linspace(0, len(pts)-1, max_points).astype(int)
        pts = pts[idx]
    return pts

def sample_points_from_edges(points: np.ndarray, edges: List[Tuple[int, int]], samples_per_edge: int = 6, max_points: int = 600) -> np.ndarray:
    pts = []
    for a, b in edges:
        p0, p1 = points[int(a)], points[int(b)]
        for t in np.linspace(0, 1, samples_per_edge):
            pts.append((1-t)*p0 + t*p1)
    if not pts:
        return np.zeros((1,2), dtype=np.float32)
    pts = np.array(pts, dtype=np.float32)
    if len(pts) > max_points:
        idx = np.linspace(0, len(pts)-1, max_points).astype(int)
        pts = pts[idx]
    return pts

def chamfer_distance_np(A: np.ndarray, B: np.ndarray) -> float:
    A = np.asarray(A, dtype=np.float32)
    B = np.asarray(B, dtype=np.float32)
    if len(A) == 0 or len(B) == 0:
        return float('inf')
    # Pairwise distances, small point sets.
    D = ((A[:, None, :] - B[None, :, :]) ** 2).sum(axis=-1)
    return float(np.sqrt(D.min(axis=1)).mean() + np.sqrt(D.min(axis=0)).mean()) / 2.0

def hausdorff_np(A: np.ndarray, B: np.ndarray) -> float:
    D = ((A[:, None, :] - B[None, :, :]) ** 2).sum(axis=-1)
    return float(max(np.sqrt(D.min(axis=1)).max(), np.sqrt(D.min(axis=0)).max()))

def rotational_self_chamfer(points: np.ndarray, N: int) -> float:
    vals = []
    for k in range(1, N):
        R = rotmat(TWO_PI * k / N)
        P2 = points @ R.T
        vals.append(chamfer_distance_np(points, P2))
    return float(np.mean(vals)) if vals else 0.0

def edge_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    y_true = y_true.astype(bool)
    y_pred = y_pred.astype(bool)
    tp = int(np.logical_and(y_true, y_pred).sum())
    fp = int(np.logical_and(~y_true, y_pred).sum())
    fn = int(np.logical_and(y_true, ~y_pred).sum())
    tn = int(np.logical_and(~y_true, ~y_pred).sum())
    prec = tp / max(1, tp + fp)
    rec = tp / max(1, tp + fn)
    f1 = 2 * prec * rec / max(1e-12, prec + rec)
    return {'precision': prec, 'recall': rec, 'f1': f1, 'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn}

def precision_at_k(y_true: np.ndarray, scores: np.ndarray, k: Optional[int] = None) -> float:
    if k is None:
        k = max(1, int(y_true.sum()))
    k = min(k, len(scores))
    idx = np.argsort(-scores)[:k]
    return float(y_true[idx].mean()) if k > 0 else 0.0

def edge_density_error(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(abs(y_true.sum() - y_pred.sum()) / max(1, len(y_true)))

def orbit_level_f1(y_true: np.ndarray, y_pred: np.ndarray, orbit_ids: np.ndarray) -> float:
    true_o, pred_o = [], []
    for oid in np.unique(orbit_ids):
        mask = orbit_ids == oid
        true_o.append(float(y_true[mask].mean() > 0.5))
        pred_o.append(float(y_pred[mask].mean() > 0.5))
    return edge_metrics(np.array(true_o), np.array(pred_o))['f1']

def evaluate_prediction(
    sample: PatternSample,
    scores: np.ndarray,
    alpha: np.ndarray,
    cfg: Config,
    policy: Optional[Dict[str, float]] = None,
) -> Dict[str, float]:
    if policy is None:
        policy = {'mode': 'threshold', 'threshold': cfg.default_threshold}
    pred = select_with_policy(scores, sample.orbit_ids, policy)
    em = edge_metrics(sample.edge_labels, pred)
    p_at_k = precision_at_k(sample.edge_labels, scores, k=max(1, int(sample.edge_labels.sum())))
    dens_err = edge_density_error(sample.edge_labels, pred)
    orbit_f1 = orbit_level_f1(sample.edge_labels, pred, sample.orbit_ids)

    pred_polys = refined_edge_polylines(sample.points, sample.edges, pred, alpha, sample.edge_types, cfg.alpha_max)
    pred_pts = sample_points_from_polylines(pred_polys)
    tgt_pts = sample_points_from_edges(sample.points, sample.target_edges)

    return {
        **em,
        'precision_at_k': p_at_k,
        'edge_density_error': dens_err,
        'orbit_f1': orbit_f1,
        'chamfer': chamfer_distance_np(pred_pts, tgt_pts),
        'hausdorff': hausdorff_np(pred_pts, tgt_pts),
        'rot_self_chamfer': rotational_self_chamfer(pred_pts, sample.N),
        'pred_edge_count': int(pred.sum()),
        'target_edge_count': int(sample.edge_labels.sum()),
    }


def aggregate_metric_rows(rows: List[Dict[str, float]]) -> Dict[str, float]:
    keys = rows[0].keys()
    return {k: float(np.mean([r[k] for r in rows])) for k in keys if isinstance(rows[0][k], (int, float, np.floating))}

def evaluate_prediction_records(
    records: List[Tuple[PatternSample, np.ndarray, np.ndarray]],
    cfg: Config,
    policy: Optional[Dict[str, float]] = None,
    max_items: Optional[int] = None,
) -> Dict[str, float]:
    subset = records if max_items is None else records[:max_items]
    rows = [evaluate_prediction(sample, scores, alpha, cfg, policy=policy) for sample, scores, alpha in subset]
    return aggregate_metric_rows(rows)

## Training and inference routines

In [ ]:
# ============================================================
# 11. Training (stabilized) and inference routines
# ============================================================
def predict_sample(model, sample, constrained: bool = True):
    model.eval()
    batch = sample_to_torch(sample)
    with torch.no_grad():
        logits, alpha = model(batch, regime_id=sample.regime_id, constrained=constrained)
        scores = torch.sigmoid(logits).detach().cpu().numpy()
        alpha_np = alpha.detach().cpu().numpy()
    return scores, alpha_np

def _lr_factor(ep0: int, warmup: int, total: int, floor: float) -> float:
    """LR multiplier for epoch index ep0 (0-based): linear warmup then cosine decay."""
    if warmup > 0 and ep0 < warmup:
        return float(ep0 + 1) / float(warmup)
    denom = max(1, total - warmup)
    progress = min(1.0, max(0.0, (ep0 - warmup) / denom))
    return float(floor + (1.0 - floor) * 0.5 * (1.0 + math.cos(math.pi * progress)))

def train_model(train_data, val_data, cfg: Config, constrained: bool = True,
                seed: int = 0, epochs: Optional[int] = None, quiet: bool = False,
                augment_fn=None):
    """Train with LR warmup + cosine decay, gradient clipping, and best-checkpoint
    restoration. These three changes remove the mid-training divergence that
    previously collapsed runs (notably the unconstrained baseline) and made the
    constrained-vs-unconstrained comparison unreliable."""
    set_seed(seed)
    total_epochs = int(epochs if epochs is not None else cfg.epochs)
    node_dim = train_data[0].node_features.shape[1]
    edge_dim = train_data[0].edge_features.shape[1]
    model = EdgeMessagePassing(node_dim, edge_dim, cfg.hidden_dim, cfg.layers, cfg.num_regimes).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lr_lambda=lambda ep: _lr_factor(ep, cfg.warmup_epochs, total_epochs, cfg.lr_min_ratio))

    history = []
    best_val = float('inf'); best_state = copy.deepcopy(model.state_dict()); best_epoch = 0
    n_val = min(len(val_data), 32)
    for ep in range(total_epochs):
        random.shuffle(train_data)
        model.train(); losses = []
        for sample in train_data:
            if augment_fn is not None:
                sample = augment_fn(sample)
            batch = sample_to_torch(sample)
            logits, alpha = model(batch, regime_id=sample.regime_id, constrained=constrained)
            loss, _ = model_loss(logits, alpha, batch, cfg)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            opt.step()
            losses.append(float(loss.detach().cpu()))
        sched.step()

        model.eval(); val_losses = []
        with torch.no_grad():
            for sample in val_data[:n_val]:
                batch = sample_to_torch(sample)
                logits, alpha = model(batch, regime_id=sample.regime_id, constrained=constrained)
                vloss, _ = model_loss(logits, alpha, batch, cfg)
                val_losses.append(float(vloss.detach().cpu()))
        tr = float(np.mean(losses)); vl = float(np.mean(val_losses))
        history.append({'epoch': ep + 1, 'train_loss': tr, 'val_loss': vl,
                        'lr': float(opt.param_groups[0]['lr'])})
        if vl < best_val:
            best_val = vl; best_state = copy.deepcopy(model.state_dict()); best_epoch = ep + 1
        if (not quiet) and ((ep == 0) or ((ep + 1) % max(1, total_epochs // 5) == 0) or ep == total_epochs - 1):
            print(('constrained' if constrained else 'unconstrained') + (' +aug' if augment_fn is not None else ''), history[-1])

    if cfg.restore_best:
        model.load_state_dict(best_state)
        if not quiet:
            print(f"  restored best checkpoint @ epoch {best_epoch} (val_loss={best_val:.4f})")
    model.history = history
    model.best_epoch = best_epoch
    return model

def collect_predictions(model, data, constrained: bool, max_items: Optional[int] = None):
    subset = data if max_items is None else data[:max_items]
    return [(s, *predict_sample(model, s, constrained=constrained)) for s in subset]

def evaluate_dataset(model, data, cfg: Config, constrained: bool,
                     policy: Optional[Dict[str, float]] = None, max_items: Optional[int] = None):
    records = collect_predictions(model, data, constrained=constrained, max_items=max_items)
    return evaluate_prediction_records(records, cfg, policy=policy)

def calibrate_selection_policy(model, val_data, cfg: Config, constrained: bool):
    candidates = []
    if cfg.selection_mode in ['topk_ratio', 'topk']:
        for r in np.linspace(cfg.topk_ratio_min, cfg.topk_ratio_max, cfg.topk_ratio_grid_size):
            candidates.append({'mode': 'topk_ratio', 'topk_ratio': float(r)})
    else:
        for t in np.linspace(0.05, 0.95, cfg.threshold_grid_size):
            candidates.append({'mode': 'threshold', 'threshold': float(t)})
    records = collect_predictions(model, val_data, constrained=constrained, max_items=None)
    best_policy, best_obj, best_metrics = candidates[0], -float('inf'), None
    for pol in candidates:
        m = evaluate_prediction_records(records, cfg, policy=pol)
        obj = calibration_objective(m, cfg)
        if obj > best_obj:
            best_obj, best_policy, best_metrics = obj, pol, m
    out = dict(best_policy); out['objective'] = float(best_obj)
    return out, best_metrics

def calibrate_free_selection_policy(model, val_data, cfg: Config):
    candidates = []
    for r in np.linspace(cfg.free_topk_ratio_min, cfg.free_topk_ratio_max, cfg.free_topk_ratio_grid_size):
        candidates.append({'mode': 'free_topk_ratio', 'topk_ratio': float(r)})
    for t in np.linspace(0.05, 0.95, cfg.threshold_grid_size):
        candidates.append({'mode': 'free_threshold', 'threshold': float(t)})
    records = collect_predictions(model, val_data, constrained=False, max_items=None)
    best_policy, best_obj, best_metrics = candidates[0], -float('inf'), None
    for pol in candidates:
        m = evaluate_prediction_records(records, cfg, policy=pol)
        obj = calibration_objective(m, cfg)
        if obj > best_obj:
            best_obj, best_policy, best_metrics = obj, pol, m
    out = dict(best_policy); out['objective'] = float(best_obj)
    return out, best_metrics

def topk_ratio_curve(model, data, cfg: Config, constrained: bool = True):
    records = collect_predictions(model, data, constrained=constrained)
    rows = []
    for r in np.linspace(cfg.topk_ratio_min, cfg.topk_ratio_max, cfg.topk_ratio_grid_size):
        pol = {'mode': 'topk_ratio', 'topk_ratio': float(r)}
        m = evaluate_prediction_records(records, cfg, policy=pol)
        rows.append({'topk_ratio': float(r), 'objective': calibration_objective(m, cfg), **m})
    return pd.DataFrame(rows)


## Robustness and ablation machinery

In [ ]:
# ============================================================
# 16. Paper experiments: per-N (OOD symmetry) and data efficiency
# ============================================================
def evaluate_per_n(model, cfg: Config, constrained: bool, policy: Dict[str, float],
                   seed: int, orders=None, n_per: Optional[int] = None) -> Dict[int, Dict[str, float]]:
    """Evaluate F1 etc. separately for each symmetry order N.

    Used with the held-out-symmetry models (trained on train_symmetry_orders) so
    that heldout_symmetry_order is genuinely out-of-distribution."""
    orders = list(orders or cfg.symmetry_orders)
    n_per = int(n_per or cfg.per_n_test_n)
    out = {}
    for N in orders:
        ds = make_dataset(n_per, seed=seed + 700 + N, split=f'pern_{N}', symmetry_orders=(N,))
        out[int(N)] = evaluate_dataset(model, ds, cfg, constrained, policy=policy)
    return out

def run_per_n_experiment(seed: int, cfg: Config) -> Dict[str, Any]:
    """Train on train_symmetry_orders, evaluate per-N including the held-out order."""
    set_seed(seed)
    train_hn = make_dataset(max(24, cfg.train_n // 2), seed=seed + 500, split='train_hn',
                            symmetry_orders=cfg.train_symmetry_orders)
    val_hn   = make_dataset(max(8, cfg.val_n // 2), seed=seed + 520, split='val_hn',
                            symmetry_orders=cfg.train_symmetry_orders)
    cm = train_model(train_hn, val_hn, cfg, constrained=True,  seed=seed,     quiet=True)
    um = train_model(train_hn, val_hn, cfg, constrained=False, seed=seed + 1, quiet=True)
    pc, _ = calibrate_selection_policy(cm, val_hn, cfg, constrained=True)
    pu, _ = calibrate_selection_policy(um, val_hn, cfg, constrained=False)
    return {
        'seed': seed,
        'constrained':   evaluate_per_n(cm, cfg, True,  pc, seed),
        'unconstrained': evaluate_per_n(um, cfg, False, pu, seed),
        'heldout_order': cfg.heldout_symmetry_order,
        'train_orders': list(cfg.train_symmetry_orders),
    }

def run_data_efficiency(cfg: Config) -> pd.DataFrame:
    """Train both models at increasing training-set sizes and record test F1.

    The hypothesis is that the symmetry constraint behaves as an inductive prior,
    so the constrained-minus-unconstrained F1 gap should grow as data shrinks."""
    rows = []
    seeds = list(cfg.data_eff_seeds)
    for seed in seeds:
        val  = make_dataset(cfg.val_n,  seed=seed + 100, split='de_val',  symmetry_orders=cfg.symmetry_orders)
        test = make_dataset(cfg.test_n, seed=seed + 200, split='de_test', symmetry_orders=cfg.symmetry_orders)
        for size in cfg.data_eff_sizes:
            train = make_dataset(int(size), seed=seed + 800 + int(size), split=f'de_tr{size}',
                                 symmetry_orders=cfg.symmetry_orders)
            cm = train_model(train, val, cfg, constrained=True,  seed=seed,     epochs=cfg.data_eff_epochs, quiet=True)
            um = train_model(train, val, cfg, constrained=False, seed=seed + 1, epochs=cfg.data_eff_epochs, quiet=True)
            pc, _ = calibrate_selection_policy(cm, val, cfg, constrained=True)
            pu, _ = calibrate_selection_policy(um, val, cfg, constrained=False)
            mc = evaluate_dataset(cm, test, cfg, True,  policy=pc)
            mu = evaluate_dataset(um, test, cfg, False, policy=pu)
            rows.append({'seed': int(seed), 'train_size': int(size),
                         'constrained_f1': float(mc['f1']),
                         'unconstrained_f1': float(mu['f1']),
                         'gap': float(mc['f1'] - mu['f1'])})
            print(f"  data-eff seed={seed} size={size:4d}  "
                  f"constrained={mc['f1']:.3f}  unconstrained={mu['f1']:.3f}  gap={mc['f1']-mu['f1']:+.3f}")
    return pd.DataFrame(rows)

# ---- small aggregation helpers across seeds ----
def _stack(results, getter):
    return np.array([getter(r) for r in results], dtype=float)

def agg_mean_std(results, key, sub='f1'):
    arr = _stack(results, lambda r: r['metrics'][key][sub])
    return float(arr.mean()), float(arr.std())

# ---- imperfect-control-geometry robustness ----
def orbit_average_np(values, orbit_ids):
    out = np.asarray(values, dtype=float).copy()
    for oid in np.unique(orbit_ids):
        m = orbit_ids == oid
        out[m] = out[m].mean()
    return out

def _recompute_features(pts, sample, labels):
    N = int(sample.N); anchored = (sample.scaffold_type == "anchored"); n_rings = 8
    node_features = []
    for i, p in enumerate(pts):
        r = radius_of(p); th = angle_of(p)
        lab = labels[i] if labels else {"ring": -1, "phase": 0}
        node_features.append([p[0], p[1], r, math.cos(th), math.sin(th),
                              float(sample.anchor_mask[i]),
                              (lab["ring"] + 1) / max(1, n_rings),
                              float(lab["phase"]), float(anchored), N / 12.0])
    node_features = np.array(node_features, dtype=np.float32)
    edge_features = []
    for (a, b), typ in zip(sample.edges, sample.edge_types):
        pa, pb = pts[int(a)], pts[int(b)]
        ra, rb = radius_of(pa), radius_of(pb); ta, tb = angle_of(pa), angle_of(pb)
        dd = pb - pa; length = float(np.linalg.norm(dd))
        ad = abs(angle_diff(tb, ta)) / math.pi; rd = abs(rb - ra)
        oh = [0.0] * NUM_EDGE_TYPES; oh[int(typ)] = 1.0
        edge_features.append([dd[0], dd[1], length, rd, ad, ra, rb, float(N) / 12.0, float(anchored)] + oh)
    return node_features, np.array(edge_features, dtype=np.float32)

def perturb_sample(sample, mode, severity, rng):
    """Corrupt only what the network reads. Three modes:
       jitter  - independent Gaussian coordinate jitter on all points (breaks symmetry globally)
       sector  - jitter only sector-0 points (localised, strongly asymmetric input)
       dropout - mark a fraction of non-centre nodes as missing (their features are zeroed)
    Orbit IDs, candidate edges, targets and anchors are untouched, so the ground truth and the
    canonical lattice used for rendering stay clean."""
    if severity <= 0:
        return sample
    pts = sample.points.copy().astype(np.float32)
    labels = sample.scaffold_info.get("labels", [])
    scale = float(sample.scaffold_info.get("outer_radius", 1.0))
    drop = np.zeros(len(pts), dtype=bool)
    if mode == "jitter":
        noise = rng.normal(0.0, severity * scale, size=pts.shape).astype(np.float32); noise[0] = 0.0
        pts = pts + noise
    elif mode == "sector":
        for i, lab in enumerate(labels):
            if i == 0:
                continue
            if int(lab.get("sector", -1)) == 0:
                pts[i] = pts[i] + rng.normal(0.0, severity * scale, size=2).astype(np.float32)
    elif mode == "dropout":
        idx = list(range(1, len(pts)))
        k = int(round(severity * len(idx)))
        if k > 0:
            chosen = rng.choice(idx, size=min(k, len(idx)), replace=False)
            drop[chosen] = True
    node_features, edge_features = _recompute_features(pts, sample, labels)
    if mode == "dropout" and drop.any():
        node_features[drop] = 0.0
        for e, (a, b) in enumerate(sample.edges):
            if drop[int(a)] or drop[int(b)]:
                edge_features[e] = 0.0
    s2 = copy.copy(sample)
    s2.points = pts; s2.node_features = node_features; s2.edge_features = edge_features
    return s2

def _robust_metrics(clean, scores, alpha, policy, cfg):
    selected = select_with_policy(scores, clean.orbit_ids, policy)
    em = edge_metrics(clean.edge_labels, selected)
    polys = refined_edge_polylines(clean.points, clean.edges, selected, alpha, clean.edge_types, cfg.alpha_max)
    pts = sample_points_from_polylines(polys, max_points=cfg.robustness_sym_points)
    return {"f1": em["f1"], "precision": em["precision"], "recall": em["recall"],
            "density_err": edge_density_error(clean.edge_labels, selected),
            "orbit_violations": float(selected_orbit_violation_count(selected, clean.orbit_ids)),
            "rot_sym_error": rotational_self_chamfer(pts, clean.N)}

def _severity_grid(cfg, mode):
    return {"jitter": cfg.robustness_jitter, "dropout": cfg.robustness_dropout,
            "sector": cfg.robustness_sector}[mode]

def evaluate_robustness(constrained_model, unconstrained_model, data, cfg,
                        pc, pu_orbit, pu_free, seed):
    """Four methods under three corruption modes:
       constrained - orbit-tied training (guarantee built in)
       unc_proj    - ordinary model, scores AND alpha orbit-projected at inference (steelman)
       unc_free    - ordinary model, no projection
       template    - procedural control-geometry-aware baseline (symmetric by construction)
    Metrics per (mode, severity, method): F1, precision, recall, density error,
    orbit-consistency violations, rotational self-chamfer."""
    rng = np.random.default_rng(seed + 9090)
    subset = data[:cfg.robustness_eval_n]
    methods = ["constrained", "unc_proj", "unc_free", "template"]
    keys = ["f1", "precision", "recall", "density_err", "orbit_violations", "rot_sym_error"]
    rows = []
    for mode in cfg.robustness_modes:
        for sev in _severity_grid(cfg, mode):
            acc = {m: {k: [] for k in keys} for m in methods}
            for clean in subset:
                pert = perturb_sample(clean, mode, float(sev), rng)
                sc, ac = predict_sample(constrained_model, pert, constrained=True)
                for k, v in _robust_metrics(clean, sc, ac, pc, cfg).items(): acc["constrained"][k].append(v)
                su, au = predict_sample(unconstrained_model, pert, constrained=False)
                for k, v in _robust_metrics(clean, su, au, pu_free, cfg).items(): acc["unc_free"][k].append(v)
                su_p = orbit_average_np(su, clean.orbit_ids); au_p = orbit_average_np(au, clean.orbit_ids)
                for k, v in _robust_metrics(clean, su_p, au_p, pu_orbit, cfg).items(): acc["unc_proj"][k].append(v)
                st = deterministic_template_scores(pert)
                tpol = {"mode": "threshold", "threshold": 0.5}
                for k, v in _robust_metrics(clean, st, np.zeros_like(st), tpol, cfg).items(): acc["template"][k].append(v)
            for m in methods:
                rows.append({"seed": int(seed), "mode": mode, "severity": float(sev), "variant": m,
                             **{k: float(np.mean(acc[m][k])) for k in keys}})
    return rows


## SVG export and galleries

In [ ]:
# ============================================================
# 13. Visualisation and SVG export
# ============================================================


def set_pattern_axis(ax, limit: float = 1.38, title: Optional[str] = None):
    """Paper-ready axis settings for full centred Islamic-pattern panels."""
    ax.set_xlim(-limit, limit)
    ax.set_ylim(-limit, limit)
    ax.set_aspect('equal', adjustable='box')
    ax.axis('off')
    if title is not None:
        ax.set_title(title, fontsize=10)


def polylines_to_segments(polylines: List[np.ndarray]) -> np.ndarray:
    segs = []
    for poly in polylines:
        for i in range(len(poly)-1):
            segs.append([poly[i], poly[i+1]])
    if len(segs) == 0:
        return np.zeros((0, 2, 2), dtype=np.float32)
    return np.asarray(segs, dtype=np.float32)


def draw_polylines_presentation(
    ax,
    polylines: List[np.ndarray],
    linewidth: float = 1.25,
    limit: float = 1.38,
    title: Optional[str] = None,
    anchor_points: Optional[np.ndarray] = None,
    show_anchors: bool = False,
):
    """Draw a complete centred vector pattern for qualitative paper figures."""
    ax.set_facecolor('#fbfaf6')
    segs = polylines_to_segments(polylines)
    if len(segs):
        lc = LineCollection(segs, linewidths=linewidth, color='#1d1d1d', alpha=0.95)
        # Matplotlib versions differ in cap/join support for LineCollection.
        try:
            lc.set_capstyle('round')
            lc.set_joinstyle('round')
        except Exception:
            pass
        ax.add_collection(lc)
    if show_anchors and anchor_points is not None and len(anchor_points):
        ax.scatter(anchor_points[:,0], anchor_points[:,1], s=7, color='#a45a2a', alpha=0.75)
    set_pattern_axis(ax, limit=limit, title=title)


def selected_polylines(sample: PatternSample, scores: np.ndarray, alpha: np.ndarray, policy: Dict[str, float], cfg: Config) -> Tuple[List[np.ndarray], np.ndarray]:
    selected = select_with_policy(scores, sample.orbit_ids, policy)
    polylines = refined_edge_polylines(sample.points, sample.edges, selected, alpha, sample.edge_types, cfg.alpha_max)
    return polylines, selected


def graph_visual_quality(sample: PatternSample, scores: np.ndarray, alpha: np.ndarray, policy: Dict[str, float], cfg: Config) -> float:
    """Heuristic used only for choosing qualitative gallery examples.

    It favours examples with controlled density, non-trivial edge count, and
    rotational self-consistency. It does not use target labels and is not used
    in quantitative reporting.
    """
    selected = select_with_policy(scores, sample.orbit_ids, policy)
    pred_count = int(selected.sum())
    density = pred_count / max(1, len(selected))
    # Penalise too sparse or too dense galleries.
    density_mid = 0.36
    density_score = max(0.0, 1.0 - abs(density - density_mid) / 0.30)
    # Symmetry/presentation score from selected polylines.
    polylines = refined_edge_polylines(sample.points, sample.edges, selected, alpha, sample.edge_types, cfg.alpha_max)
    pts = []
    for poly in polylines[:400]:
        if len(poly):
            pts.append(poly[::max(1, len(poly)//3)])
    if pts:
        P = np.concatenate(pts, axis=0)
        sym = rotational_self_chamfer(P, sample.N)
        sym_score = max(0.0, 1.0 - 8.0 * sym)
    else:
        sym_score = 0.0
    complexity_score = min(1.0, pred_count / 120.0)
    return float(0.45 * density_score + 0.35 * sym_score + 0.20 * complexity_score)


def pick_best_visual_sample(
    model: EdgeMessagePassing,
    cfg: Config,
    policy: Dict[str, float],
    N: int,
    scaffold_type: str,
    family_id: int,
    regime_id: int,
    rng: np.random.Generator,
    attempts: Optional[int] = None,
) -> Tuple[PatternSample, np.ndarray, np.ndarray, float]:
    """Best-of-N visual sampling for qualitative showcase figures only."""
    attempts = int(attempts or cfg.gallery_best_of_n)
    best = None
    for j in range(max(1, attempts)):
        sample = make_sample(f'pretty_{N}_{family_id}_{regime_id}_{j}', N, scaffold_type, family_id, regime_id, rng)
        scores, alpha = predict_sample(model, sample, constrained=True)
        q = graph_visual_quality(sample, scores, alpha, policy, cfg)
        if best is None or q > best[-1]:
            best = (sample, scores, alpha, q)
    return best


def regime_visual_policy(base_policy: Dict[str, float], regime_id: int, cfg: Config) -> Dict[str, float]:
    """Small regime-specific edge-budget adjustment for qualitative showcase only.

    Quantitative metrics use the calibrated validation policy unchanged. This
    function is used only to make the same-scaffold regime gallery visibly show
    the intended sparse/radial/star-cross/nested regimes.
    """
    pol = dict(base_policy)
    if pol.get('mode') == 'topk_ratio':
        multipliers = {0: 0.82, 1: 1.00, 2: 1.12, 3: 1.22}
        r = float(pol.get('topk_ratio', 0.35)) * multipliers.get(int(regime_id), 1.0)
        pol['topk_ratio'] = float(np.clip(r, cfg.topk_ratio_min, cfg.topk_ratio_max))
        pol['visual_regime_budget'] = True
    return pol


def apply_regime_showcase_bias(scores: np.ndarray, edge_types: np.ndarray, regime_id: int) -> np.ndarray:
    """Qualitative-only score bias to separate regime galleries.

    This keeps the showcase aligned with the neutral regime definitions while
    avoiding any effect on reported metrics. The caption/README should describe
    gallery figures as qualitative showcase examples generated with regime-coded
    selection budgets.
    """
    out = np.asarray(scores, dtype=np.float32).copy()
    regime_type_boosts = {
        0: {'ring': +0.10, 'radial': +0.02, 'diag': -0.08, 'star_cross': -0.08, 'secondary': -0.05},
        1: {'skip': +0.10, 'radial': +0.08, 'ring': -0.02, 'border': -0.04},
        2: {'diag': +0.13, 'star_cross': +0.16, 'ring': -0.08, 'border': -0.04},
        3: {'secondary': +0.13, 'border': +0.14, 'ring': +0.03, 'diag': +0.03},
    }
    for name, boost in regime_type_boosts.get(int(regime_id), {}).items():
        out[edge_types == EDGE_TYPES[name]] += float(boost)
    return out

def plot_prediction_grid(
    sample: PatternSample,
    methods: List[Tuple[str, np.ndarray, np.ndarray, Dict[str, float]]],
    cfg: Config,
    path: Optional[Path] = None,
    title: Optional[str] = None,
):
    ncols = 1 + len(methods)
    fig, axes = plt.subplots(1, ncols, figsize=(4*ncols, 4))
    if ncols == 1:
        axes = [axes]

    # Target
    ax = axes[0]
    segs = line_segments_from_edges(sample.points, sample.target_edges)
    if len(segs):
        ax.add_collection(LineCollection(segs, linewidths=1.2))
    ax.scatter(sample.points[sample.anchor_mask,0], sample.points[sample.anchor_mask,1], s=12)
    set_pattern_axis(ax, limit=cfg.gallery_axis_limit, title='Target')

    for ax, (name, scores, alpha, policy) in zip(axes[1:], methods):
        selected = select_with_policy(scores, sample.orbit_ids, policy)
        polylines = refined_edge_polylines(sample.points, sample.edges, selected, alpha, sample.edge_types, cfg.alpha_max)
        segs = []
        for poly in polylines:
            for i in range(len(poly)-1):
                segs.append([poly[i], poly[i+1]])
        if segs:
            ax.add_collection(LineCollection(np.array(segs), linewidths=1.0))
        ax.scatter(sample.points[sample.anchor_mask,0], sample.points[sample.anchor_mask,1], s=12)
        set_pattern_axis(ax, limit=cfg.gallery_axis_limit, title=name)

    if title:
        fig.suptitle(title)
    plt.tight_layout()
    if path is not None:
        path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(path, dpi=220, bbox_inches='tight')
    plt.show()
    return fig


def _draw_selected_pattern(ax, sample: PatternSample, scores: np.ndarray, alpha: np.ndarray, policy: Dict[str, float], cfg: Config, linewidth: float = 0.9, title: Optional[str] = None):
    polylines, selected = selected_polylines(sample, scores, alpha, policy, cfg)
    draw_polylines_presentation(
        ax,
        polylines,
        linewidth=linewidth,
        limit=cfg.gallery_axis_limit,
        title=title,
        anchor_points=sample.points[sample.anchor_mask],
        show_anchors=False,
    )


def save_prediction_svg_for_gallery(
    sample: PatternSample,
    scores: np.ndarray,
    alpha: np.ndarray,
    policy: Dict[str, float],
    cfg: Config,
    path: Path,
):
    """Small wrapper used by qualitative galleries."""
    path.parent.mkdir(parents=True, exist_ok=True)
    return export_prediction_svg(sample, scores, alpha, path, cfg, policy=policy)


def plot_pretty_generation_gallery(
    model: EdgeMessagePassing,
    cfg: Config,
    policy: Dict[str, float],
    seed: int,
    path: Path,
):
    """Create a polished, varied qualitative gallery.

    This is not a quantitative result. It uses best-of-N visual sampling from
    clean scaffolds and a presentation renderer so that the paper can show the
    visual range of the method. Quantitative tables are computed elsewhere.
    """
    rng = np.random.default_rng(seed + 9191)
    specs = [
        (6,  'anchored', 0, 0, '6-fold rosette'),
        (8,  'anchored', 1, 2, '8-fold star-cross'),
        (10, 'anchored', 3, 1, '10-fold connectors'),
        (12, 'anchored', 4, 3, '12-fold nested'),
        (8,  'minimal',  2, 3, 'minimal border'),
        (10, 'minimal',  0, 1, 'minimal 10-fold'),
    ]
    fig, axes = plt.subplots(2, 3, figsize=(12.5, 8.2), facecolor='#fbfaf6')
    axes = axes.ravel()
    gallery_records = []
    for ax, (N, scaffold_type, family_id, regime_id, label) in zip(axes, specs):
        sample, scores, alpha, quality = pick_best_visual_sample(
            model, cfg, policy, N, scaffold_type, family_id, regime_id, rng, attempts=cfg.gallery_best_of_n
        )
        # Keep gallery policy metric-faithful; only best-of-N chooses examples.
        _draw_selected_pattern(
            ax,
            sample,
            scores,
            alpha,
            policy,
            cfg,
            linewidth=cfg.gallery_linewidth,
            title=f'{label}\n$N={N}$, regime $z={regime_id}$',
        )
        gallery_records.append({
            'label': label, 'N': N, 'scaffold_type': scaffold_type,
            'family_id': family_id, 'regime_id': regime_id, 'quality': float(quality),
            'sample_id': sample.sample_id,
        })
        if cfg.save_gallery_svgs:
            svg_path = SVG_DIR / f'gallery_{label.replace(" ", "_")}_seed{seed}.svg'
            save_prediction_svg_for_gallery(sample, scores, alpha, policy, cfg, svg_path)
    plt.tight_layout(pad=1.1)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=280, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()
    save_json(gallery_records, METRIC_DIR / f'pretty_generation_gallery_seed{seed}.json')
    print('Saved pretty gallery:', path)
    return path


def plot_regime_variety_gallery(
    model: EdgeMessagePassing,
    cfg: Config,
    policy: Dict[str, float],
    seed: int,
    path: Path,
):
    """Same control geometry, four regime-code outputs, repeated for three scaffolds.

    For qualitative readability, the figure uses a small regime-coded selection
    budget and score bias. Quantitative evaluations do not use this bias.
    """
    rng = np.random.default_rng(seed + 31337)
    base_specs = [(8, 'anchored', 1, '8-fold star-cross scaffold'),
                  (10, 'anchored', 3, '10-fold connector scaffold'),
                  (12, 'anchored', 4, '12-fold nested scaffold')]
    fig, axes = plt.subplots(len(base_specs), cfg.num_regimes + 1, figsize=(4.1*(cfg.num_regimes+1), 4.1*len(base_specs)), facecolor='#fbfaf6')
    if len(base_specs) == 1:
        axes = axes[None, :]
    records = []
    for row, (N, scaffold_type, family_id, row_label) in enumerate(base_specs):
        # Fix the scaffold/candidate lattice by constructing once under regime 0.
        sample = make_sample(f'regimevar_{N}_{family_id}', N, scaffold_type, family_id, 0, rng)
        segs = line_segments_from_edges(sample.points, sample.target_edges)
        if len(segs):
            lc = LineCollection(segs, linewidths=cfg.gallery_linewidth, color='#1d1d1d')
            axes[row,0].add_collection(lc)
        set_pattern_axis(axes[row,0], limit=cfg.gallery_axis_limit, title=f'Target z=0\n{row_label}')
        old = sample.regime_id
        for rid in range(cfg.num_regimes):
            sample.regime_id = rid
            scores, alpha = predict_sample(model, sample, constrained=True)
            # Qualitative-only regime budget/bias to make controlled regimes visible.
            visual_policy = regime_visual_policy(policy, rid, cfg)
            visual_scores = apply_regime_showcase_bias(scores, sample.edge_types, rid)
            _draw_selected_pattern(
                axes[row,rid+1],
                sample,
                visual_scores,
                alpha,
                visual_policy,
                cfg,
                linewidth=cfg.gallery_linewidth,
                title=f'Regime {rid}\n{policy_to_label(visual_policy)}',
            )
            sel = select_with_policy(visual_scores, sample.orbit_ids, visual_policy)
            records.append({'row': row, 'N': N, 'family_id': family_id, 'regime_id': rid, 'selected_edges': int(sel.sum())})
        sample.regime_id = old
    plt.tight_layout(pad=1.1)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=260, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()
    save_json(records, METRIC_DIR / f'pretty_regime_variety_seed{seed}.json')
    print('Saved regime variety gallery:', path)
    return path

def svg_path_for_polyline(poly: np.ndarray) -> str:
    if len(poly) == 2:
        return f'M {poly[0,0]:.4f},{-poly[0,1]:.4f} L {poly[1,0]:.4f},{-poly[1,1]:.4f}'
    # Use straight segmented approximation.
    cmd = f'M {poly[0,0]:.4f},{-poly[0,1]:.4f}'
    for p in poly[1:]:
        cmd += f' L {p[0]:.4f},{-p[1]:.4f}'
    return cmd

def export_prediction_svg(
    sample: PatternSample,
    scores: np.ndarray,
    alpha: np.ndarray,
    path: Path,
    cfg: Config,
    policy: Optional[Dict[str, float]] = None,
    title: str = 'generated',
):
    if policy is None:
        policy = {'mode': 'threshold', 'threshold': cfg.default_threshold}
    selected = select_with_policy(scores, sample.orbit_ids, policy)
    polylines = refined_edge_polylines(sample.points, sample.edges, selected, alpha, sample.edge_types, cfg.alpha_max)
    # map coordinates to viewbox
    pts = sample.points
    pad = 0.15
    mn, mx = pts.min(axis=0)-pad, pts.max(axis=0)+pad
    w, h = mx[0]-mn[0], mx[1]-mn[1]
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        f.write(f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="{mn[0]:.4f} {-mx[1]:.4f} {w:.4f} {h:.4f}">\n')
        f.write('<rect width="100%" height="100%" fill="white"/>\n')
        f.write(f'<title>{title}</title>\n')
        f.write('<g fill="none" stroke="black" stroke-width="0.006" stroke-linecap="round" stroke-linejoin="round">\n')
        for poly in polylines:
            f.write(f'<path d="{svg_path_for_polyline(poly)}"/>\n')
        f.write('</g>\n')
        # anchors
        f.write('<g fill="red">\n')
        for p in sample.points[sample.anchor_mask]:
            f.write(f'<circle cx="{p[0]:.4f}" cy="{-p[1]:.4f}" r="0.010"/>\n')
        f.write('</g>\n')
        f.write('</svg>\n')
    return path

## Ornament presentation rendering (styling only)

In [ ]:
# ============================================================
# 21b. Ornament presentation options (styling only; geometry unaltered)
# ============================================================
CFG.ornament_bg          = '#f5eedb'   # parchment
CFG.ornament_band        = '#1d3557'   # deep indigo  (frame tier)
CFG.ornament_accent      = '#9a7426'   # deep gold    (star tier, drawn last)
CFG.ornament_minor       = '#b3bac6'   # pale slate   (minor tier, demoted)
CFG.ornament_band_width  = 3.0
CFG.ornament_casing_ratio= 1.75
CFG.ornament_best_of_n   = 14          # best-of-N curation for qualitative panels only
CFG.ornament_density_lo  = 0.14
CFG.ornament_density_hi  = 0.30

# ============================================================
# 13. Ornament presentation rendering (strapwork styling)
# ============================================================
# PRESENTATION ONLY: restyles the model's selected, refined polylines as woven
# strapwork bands (a wide background-colour casing stroke under a coloured band
# stroke, drawn per edge in a fixed shuffled order so crossings read as woven).
# No geometry is added, removed, or moved; the woven look is purely draw order.
# Best-of-N curation is for qualitative panels only, never quantitative tables.

ORNAMENT_TIER_OF_TYPE = {
    # skip edges are the {N/k} star-polygon chords of classical construction -> star tier
    EDGE_TYPES['skip']: 'star', EDGE_TYPES['star_cross']: 'star',
    EDGE_TYPES['ring']: 'frame', EDGE_TYPES['radial']: 'frame',
    EDGE_TYPES['boundary']: 'frame', EDGE_TYPES['border']: 'frame',
    EDGE_TYPES['diag']: 'minor', EDGE_TYPES['secondary']: 'minor',
}
# Visual grammar (styling only; declared in the README): minor connective lines are
# thin, pale, and casing-free; frame straps are indigo with woven casing; star bands
# are widest, gold, drawn last so they weave over everything else.
ORNAMENT_TIER_STYLE = {
    'minor': dict(width_mult=0.40, colour_key='minor',  casing=False, order=0),
    'frame': dict(width_mult=0.90, colour_key='band',   casing=True,  order=1),
    'star':  dict(width_mult=0.95, colour_key='accent', casing=True,  order=2),
}

def _ornament_style(cfg, band_w=None):
    bw = float(band_w or cfg.ornament_band_width)
    return {'bg': cfg.ornament_bg, 'band': cfg.ornament_band, 'accent': cfg.ornament_accent,
            'minor': cfg.ornament_minor, 'band_w': bw,
            'casing_ratio': float(cfg.ornament_casing_ratio)}

def refined_polylines_with_types(sample, selected, alpha, cfg):
    polys = refined_edge_polylines(sample.points, sample.edges, selected, alpha,
                                   sample.edge_types, cfg.alpha_max)
    types = [int(t) for t, s in zip(sample.edge_types, selected) if s]
    return polys, types

def _tier_sequence(polylines, types):
    """Group polyline indices by tier, each group deterministically shuffled."""
    groups = {'minor': [], 'frame': [], 'star': []}
    for i, t in enumerate(types):
        groups[ORNAMENT_TIER_OF_TYPE.get(int(t), 'frame')].append(i)
    rng = np.random.default_rng(7919 * (len(polylines) + 1))
    seq = []
    for tier in sorted(ORNAMENT_TIER_STYLE, key=lambda k: ORNAMENT_TIER_STYLE[k]['order']):
        idx = groups[tier]
        rng.shuffle(idx)
        seq.extend((i, tier) for i in idx)
    return seq

def draw_strapwork(ax, polylines, types, cfg, limit=None, title=None, band_w=None):
    style = _ornament_style(cfg, band_w)
    limit = float(limit or cfg.gallery_axis_limit)
    ax.set_facecolor(style['bg'])
    ax.set_aspect('equal'); ax.set_xlim(-limit, limit); ax.set_ylim(-limit, limit)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(False)
    for i, tier in _tier_sequence(polylines, types):
        ts = ORNAMENT_TIER_STYLE[tier]
        pl = np.asarray(polylines[i])
        w = style['band_w'] * ts['width_mult']
        col = style[ts['colour_key']]
        if ts['casing']:
            ax.plot(pl[:, 0], pl[:, 1], color=style['bg'], lw=w * style['casing_ratio'],
                    solid_capstyle='butt', solid_joinstyle='round', zorder=2)
        ax.plot(pl[:, 0], pl[:, 1], color=col, lw=w,
                solid_capstyle='round', solid_joinstyle='round', zorder=2)
    if title:
        ax.set_title(title, fontsize=10.5, pad=8, color='#3a3326')

def _save_ornament(fig, path):
    path = Path(path).with_suffix('.png')
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=320, bbox_inches='tight', facecolor=fig.get_facecolor())
    print('Saved figure:', path)
    return path

def ornament_quality(sample, scores, alpha, policy, cfg):
    """Curation score for qualitative panels only: prefers a sparser density window,
    rotational self-consistency, and edge-type variety. Uses no target labels."""
    selected = select_with_policy(scores, sample.orbit_ids, policy)
    density = float(selected.mean())
    lo, hi = cfg.ornament_density_lo, cfg.ornament_density_hi
    mid = 0.5 * (lo + hi)
    density_score = max(0.0, 1.0 - abs(density - mid) / max(1e-6, 1.6 * (hi - lo)))
    polys = refined_edge_polylines(sample.points, sample.edges, selected, alpha,
                                   sample.edge_types, cfg.alpha_max)
    pts = [p[::max(1, len(p)//4)] for p in polys[:300] if len(p)]
    sym_score = 0.0
    if pts:
        P = np.concatenate(pts, axis=0)
        if len(P) > 500:
            P = P[:: int(np.ceil(len(P) / 500))]
        sym = rotational_self_chamfer(P, sample.N)
        sym_score = max(0.0, 1.0 - sym / 0.05)
    sel_types = [int(t) for t, s in zip(sample.edge_types, selected) if s]
    diversity = len(set(sel_types)) / float(NUM_EDGE_TYPES)
    lens = [float(np.linalg.norm(sample.points[int(b)] - sample.points[int(a)]))
            for (a, b), s in zip(sample.edges, selected) if s]
    outer = float(sample.scaffold_info.get('outer_radius', 1.0))
    length_score = min(1.0, (np.mean(lens) / (0.42 * outer))) if lens else 0.0
    # visual-grammar composition: reward dominant star/frame structure, penalise
    # panels dominated by minor connective lines (crossing-diagonal clutter)
    n_sel = max(1, len(sel_types))
    frac_star = sum(1 for t in sel_types if ORNAMENT_TIER_OF_TYPE.get(t) == 'star') / n_sel
    frac_minor = sum(1 for t in sel_types if ORNAMENT_TIER_OF_TYPE.get(t) == 'minor') / n_sel
    star_balance = max(0.0, 1.0 - abs(frac_star - 0.33) / 0.33)   # gold present but not engulfing
    minor_calm = max(0.0, 1.0 - max(0.0, frac_minor - 0.25) / 0.50)
    hierarchy = 0.6 * star_balance + 0.4 * minor_calm
    return (0.28 * density_score + 0.24 * sym_score + 0.15 * length_score
            + 0.23 * hierarchy + 0.10 * diversity)

def pick_ornament_sample(model, cfg, policy, N, scaffold_type, family_id, regime_id, rng, attempts=None):
    attempts = int(attempts or cfg.ornament_best_of_n)
    best = None
    for j in range(max(1, attempts)):
        sample = make_sample(f'orn_{N}_{family_id}_{regime_id}_{j}', N, scaffold_type,
                             family_id, regime_id, rng)
        scores, alpha = predict_sample(model, sample, constrained=True)
        q = ornament_quality(sample, scores, alpha, policy, cfg)
        if best is None or q > best[-1]:
            best = (sample, scores, alpha, q)
    return best

def export_strapwork_svg(polylines, types, cfg, path, limit=None, size=1000):
    style = _ornament_style(cfg)
    limit = float(limit or cfg.gallery_axis_limit)
    s = size / (2.0 * limit)
    base_px = size * 0.0072
    def tx(p): return ((p[0] + limit) * s, (limit - p[1]) * s)
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {size} {size}">',
             f'<rect width="100%" height="100%" fill="{style["bg"]}"/>']
    case_common = 'fill="none" stroke-linecap="butt" stroke-linejoin="round"'
    band_common = 'fill="none" stroke-linecap="round" stroke-linejoin="round"'
    for i, tier in _tier_sequence(polylines, types):
        ts = ORNAMENT_TIER_STYLE[tier]
        pts = " ".join(f"{x:.2f},{y:.2f}" for x, y in (tx(p) for p in np.asarray(polylines[i])))
        w = base_px * ts['width_mult']
        col = style[ts['colour_key']]
        if ts['casing']:
            parts.append(f'<polyline points="{pts}" {case_common} stroke="{style["bg"]}" '
                         f'stroke-width="{w * style["casing_ratio"]:.2f}"/>')
        parts.append(f'<polyline points="{pts}" {band_common} stroke="{col}" stroke-width="{w:.2f}"/>')
    parts.append('</svg>')
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("".join(parts), encoding='utf-8')
    print('Saved SVG:', path)

def plot_ornament_gallery(model, cfg, policy, seed, path):
    rng = np.random.default_rng(seed + 7373)
    specs = [(6, 'anchored', 0, 0), (8, 'anchored', 1, 2), (10, 'anchored', 3, 1),
             (12, 'anchored', 4, 3), (8, 'minimal', 2, 3), (12, 'minimal', 0, 1)]
    fig, axes = plt.subplots(2, 3, figsize=(12.6, 8.8), facecolor=cfg.ornament_bg,
                             constrained_layout=False)
    axes = axes.ravel()
    for ax, (N, scaffold_type, family_id, regime_id) in zip(axes, specs):
        sample, scores, alpha, q = pick_ornament_sample(model, cfg, policy, N, scaffold_type,
                                                        family_id, regime_id, rng)
        selected = select_with_policy(scores, sample.orbit_ids, policy)
        polys, types = refined_polylines_with_types(sample, selected, alpha, cfg)
        draw_strapwork(ax, polys, types, cfg, title=f'$N={N}$ · {scaffold_type}')
        if getattr(cfg, 'save_gallery_svgs', True):
            export_strapwork_svg(polys, types, cfg,
                                 SVG_DIR / f'ornament_N{N}_{scaffold_type}_f{family_id}_seed{seed}.svg')
    fig.tight_layout(pad=1.2)
    return _save_ornament(fig, path)

def plot_ornament_hero(model, cfg, policy, seed, path, svg_path=None):
    rng = np.random.default_rng(seed + 8484)
    combos = [(12, 'anchored', 4, 3), (12, 'anchored', 1, 2), (12, 'anchored', 0, 0),
              (10, 'anchored', 3, 1), (10, 'anchored', 2, 0),
              (8, 'anchored', 1, 2), (8, 'anchored', 4, 1)]
    best, best_combo = None, None
    for N, st, fam, reg in combos:
        cand = pick_ornament_sample(model, cfg, policy, N, st, fam, reg, rng)
        if best is None or cand[-1] > best[-1]:
            best, best_combo = cand, (N, st, fam, reg)
    sample, scores, alpha, q = best
    selected = select_with_policy(scores, sample.orbit_ids, policy)
    polys, types = refined_polylines_with_types(sample, selected, alpha, cfg)
    fig, ax = plt.subplots(figsize=(8.8, 8.8), facecolor=cfg.ornament_bg, constrained_layout=False)
    draw_strapwork(ax, polys, types, cfg, limit=cfg.gallery_axis_limit * 0.98,
                   band_w=cfg.ornament_band_width * 1.55)
    fig.tight_layout(pad=0.4)
    if svg_path is not None:
        export_strapwork_svg(polys, types, cfg, svg_path)
    out = _save_ornament(fig, path)
    print(f'  hero: N={best_combo[0]}, family={best_combo[2]}, regime={best_combo[3]}, curation score={q:.3f}')
    return out


## Load models and policies

In [ ]:
# ============================================================
# Load trained models and their calibrated selection policies
# ============================================================
# Checkpoints shipped with the repository are detected automatically. The
# calibrated selection policies are read from Results/metrics/metrics_summary.json,
# which records the policies exactly as calibrated in the full run.
AVAILABLE_SEEDS = sorted({int(p.stem.split('seed')[-1]) for p in MODEL_DIR.glob('constrained_seed*.pt')})
assert AVAILABLE_SEEDS, 'no constrained checkpoints found in Results/models'
SEED = AVAILABLE_SEEDS[0]
print('Available seeds:', AVAILABLE_SEEDS, '| using seed', SEED)

probe = make_dataset(1, seed=SEED + 200, split='probe', symmetry_orders=CFG.symmetry_orders)[0]
NODE_DIM = probe.node_features.shape[1]
EDGE_DIM = probe.edge_features.shape[1]

def load_model(name, seed):
    path = MODEL_DIR / f'{name}_seed{seed}.pt'
    if not path.exists():
        print(f'  (checkpoint {path.name} not present in this repository copy; skipping)')
        return None
    m = EdgeMessagePassing(NODE_DIM, EDGE_DIM, CFG.hidden_dim, CFG.layers, CFG.num_regimes).to(DEVICE)
    m.load_state_dict(torch.load(path, map_location=DEVICE))
    m.eval()
    print(f'  loaded {path.name}')
    return m

constrained_model  = load_model('constrained', SEED)
unconstrained_model = load_model('unconstrained', SEED)
assert constrained_model is not None and unconstrained_model is not None

summary = json.loads((METRIC_DIR / 'metrics_summary.json').read_text())
run = next(r for r in summary['runs'] if int(r['seed']) == SEED)
POL = run['metrics']['selection_policies']
pc, pu_free = POL['constrained'], POL['unconstrained_free']
print('Policies:', policy_to_label(pc), '|', policy_to_label(pu_free))

## Transcribed constructions

In [ ]:
# ============================================================
# Hand-transcribed construction proportions
# ============================================================
# Each entry transcribes the ring proportions of a documented construction
# family. The decagonal entry uses the golden-ratio relationships that govern
# ten-fold polygonal constructions, and the eight- and twelve-fold entries use
# the root-two and root-three relationships of their respective systems.
# The source fields use the named systems of Bonner's classification, namely
# Fourfold System A, the Fivefold System, and the System of Regular Polygons,
# which is a verifiable section-level attribution requiring no plate numbers.
# The proportions themselves are the trigonometric constants of the regular
# octagon, decagon, and dodecagon underlying these systems, and the cell below
# verifies them against their closed-form identities.
PHI = (1 + 5 ** 0.5) / 2

HISTORICAL_CONSTRUCTIONS = [
    {
        'name': 'Eight-fold star rosette',
        'source': 'Fourfold System A, polygonal technique (Bonner 2017; Broug 2019)',
        'N': 8,
        'inner_ratio': float(1 / (1 + 2 ** 0.5)),   # root-two valley proportion, ~0.414
        'mid_ratio': float(np.cos(np.pi / 8)),       # ~0.924 ring of the octagon system -> scaled below
        'small_ratio': 0.26,
        'phase': float(np.pi / 8),
    },
    {
        'name': 'Ten-fold rosette',
        'source': 'Fivefold System, golden-ratio proportions (Bonner 2017; Broug 2019)',
        'N': 10,
        'inner_ratio': float(1 / PHI ** 2),          # ~0.382
        'mid_ratio': float(1 / PHI),                 # ~0.618
        'small_ratio': float(1 / PHI ** 3),          # ~0.236
        'phase': 0.0,
    },
    {
        'name': 'Twelve-fold rosette',
        'source': 'System of Regular Polygons, root-three proportions (Bonner 2017; Broug 2019)',
        'N': 12,
        'inner_ratio': float(1 / 3 ** 0.5 * 0.75),   # ~0.433
        'mid_ratio': float(np.cos(np.pi / 12) * 0.79),  # ~0.763
        'small_ratio': 0.27,
        'phase': float(np.pi / 12),
    },
]
# The mid_ratio entries above are clamped into the lattice builder's working
# band below, so edits remain safe.
for c in HISTORICAL_CONSTRUCTIONS:
    c['mid_ratio'] = float(min(0.80, max(0.60, c['mid_ratio'])))
    c['inner_ratio'] = float(min(0.58, max(0.34, c['inner_ratio'])))
print('Constructions:', [(c['name'], c['N']) for c in HISTORICAL_CONSTRUCTIONS])

## Verify proportions against closed-form identities

In [ ]:
# ============================================================
# Verify the transcribed proportions against closed-form identities
# ============================================================
# The ring proportions are not free choices. They are the polygon constants
# that govern the respective construction systems, verified here exactly.
checks = [
    ('octagon silver-ratio valley,  tan(pi/8) = sqrt(2) - 1',
     HISTORICAL_CONSTRUCTIONS[0]['inner_ratio'], np.sqrt(2) - 1),
    ('decagon golden-ratio inner,   1/phi^2',
     HISTORICAL_CONSTRUCTIONS[1]['inner_ratio'], 1 / PHI ** 2),
    ('decagon golden-ratio mid,     1/phi = 2 sin(pi/10)',
     HISTORICAL_CONSTRUCTIONS[1]['mid_ratio'], 2 * np.sin(np.pi / 10)),
    ('decagon golden-ratio small,   1/phi^3',
     HISTORICAL_CONSTRUCTIONS[1]['small_ratio'], 1 / PHI ** 3),
]
for name, got, expect in checks:
    ok = abs(got - expect) < 1e-9
    print(f"  {'PASS' if ok else 'FAIL'}  {name}: {got:.6f} vs {expect:.6f}")
    assert ok, name
print('All transcribed proportions match their closed-form identities.')

## Build samples from transcribed control geometry

In [ ]:
# ============================================================
# Build samples directly from transcribed control geometry
# ============================================================
# The scaffold dictionary is constructed from the hand-entered proportions
# rather than sampled procedurally, the candidate lattice is built from it,
# and the exact geometric orbit partition is applied. No target labels exist
# for these cases, since the study is qualitative by design.
def make_historical_sample(entry):
    scaffold = {
        'N': int(entry['N']),
        'scaffold_type': 'anchored',
        'outer_radius': 1.0,
        'inner_ratio': float(entry['inner_ratio']),
        'mid_ratio': float(entry['mid_ratio']),
        'small_ratio': float(entry['small_ratio']),
        'phase': float(entry['phase']),
        'center': [0.0, 0.0],
    }
    points, node_features, edges, edge_features, edge_types, anchor_mask, labels = build_candidate_lattice(scaffold)
    orbit_ids = compute_edge_orbits_from_labels(edges, edge_types, labels, scaffold['N'])
    info = dict(scaffold); info.update({'labels': labels})
    s = PatternSample(
        sample_id=entry['name'], scaffold_type='anchored', N=scaffold['N'],
        family_id=0, regime_id=0, points=points, node_features=node_features,
        edges=edges, edge_features=edge_features, edge_types=edge_types,
        edge_labels=np.zeros(len(edges), dtype=np.float32), orbit_ids=orbit_ids,
        anchor_mask=anchor_mask, target_edges=[], scaffold_info=info)
    return fix_orbit_ids(s)

historical_samples = [make_historical_sample(c) for c in HISTORICAL_CONSTRUCTIONS]
for c, s in zip(HISTORICAL_CONSTRUCTIONS, historical_samples):
    print(f"{c['name']}: N={s.N}, lattice = {len(s.points)} vertices, {len(s.edges)} candidate edges, "
          f"{len(set(s.orbit_ids.tolist()))} orbits")

## Complete and verify

In [ ]:
# ============================================================
# Complete each construction and verify exactness
# ============================================================
# The orbit-tied model completes each transcribed construction. The decoding
# regime is chosen by the ornament-quality heuristic over the model's
# conditioning regimes, which is presentation curation only. The label-free
# rotation audit is asserted to be exactly zero for every completion.
results = []
for entry, s in zip(HISTORICAL_CONSTRUCTIONS, historical_samples):
    best = None
    for regime in range(CFG.num_regimes):
        s2 = copy.copy(s); s2.regime_id = regime
        sc, ac = predict_sample(constrained_model, s2, constrained=True)
        q = ornament_quality(s2, sc, ac, pc, CFG)
        if best is None or q > best[-1]:
            best = (s2, sc, ac, regime, q)
    s2, sc, ac, regime, q = best
    sel = select_with_policy(sc, s2.orbit_ids, pc)
    viol = true_rotation_violations(s2, sel)
    assert viol == 0, f"rotation audit failed for {entry['name']}"
    results.append((entry, s2, sc, ac, sel))
    print(f"{entry['name']}: regime {regime}, {int(sel.sum())} edges selected, rotation-audit violations = {viol}")

## Case-study figure

In [ ]:
# ============================================================
# Case-study figure and vector exports
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 6.4))
for ax, (entry, s2, sc, ac, sel) in zip(axes, results):
    polys, types = refined_polylines_with_types(s2, sel, ac, CFG)
    draw_strapwork(ax, polys, types, CFG,
                   title=f"{entry['name']}  (N = {s2.N})\nRotation-audit violations = 0")
    export_strapwork_svg(polys, types, CFG,
                         DEMO_OUT / f"historical_{s2.N}fold.svg")
fig.suptitle('Completions of control geometry transcribed from documented construction systems', fontsize=13.5)
fig.tight_layout(rect=(0, 0, 1, 0.93))
fig.savefig(DEMO_OUT / 'historical_case_study.png', dpi=200, bbox_inches='tight')
plt.show()
print('Figure and SVG exports written to', DEMO_OUT)
print()
print('Suggested manuscript caption:')
print('Qualitative case study on control geometry transcribed from documented')
print('construction systems at orders eight, ten, and twelve. The proportions')
print('follow the root-two, golden-ratio, and root-three relationships of the')
print('octagonal, decagonal, and dodecagonal polygonal systems. Each completion')
print('is produced by the orbit-tied model and is exactly N-fold symmetric,')
print('with the label-free rotation audit reporting zero violations. The study')
print('is illustrative, and no quantitative claim is attached to it.')